In [0]:
%pip install transformers torch accelerate

In [0]:
from huggingface_hub import login

login()

In [0]:
# Purpose: Generate month 5 synthetic auto loans, double new defaults for used Europa cars, and score account-level ECL with models trained on months 1-2.
# Refactored: no directories or CSV files are created; all outputs are kept as pandas DataFrames.

import threading
import time
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 20261120
BRANDS = ["Io", "Europa", "Ganymede", "Callisto", "Amalthea"]
DATES = pd.to_datetime(["2026-01-31", "2026-02-28", "2026-03-31", "2026-04-30", "2026-05-31"])


def heartbeat(stop):
    elapsed = 0
    while not stop.wait(5):
        elapsed += 5
        print(f"[heartbeat] still working - {elapsed}s elapsed; month 5 generation and scoring are progressing", flush=True)


def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))


def generate_snapshots(n=6000):
    rng = np.random.default_rng(SEED)
    condition = rng.choice(["New", "Used"], n, p=[0.35, 0.65])
    price = np.where(condition == "New", rng.normal(4500, 450, n), rng.normal(300, 55, n)).clip(100)
    term = rng.choice([12, 24, 36, 48, 60], n, p=[0.08, 0.17, 0.30, 0.27, 0.18])
    starts = pd.Timestamp("2021-01-01") + pd.to_timedelta(rng.integers(0, 1809, n), unit="D")
    deposit = rng.uniform(0, 0.35, n)
    base = pd.DataFrame({
        "account_id": [f"MOON-{i:06d}" for i in range(1, n + 1)],
        "car_brand": rng.choice(BRANDS, n), "vehicle_condition": condition,
        "vehicle_price": price.round(2), "loan_start_date": starts,
        "original_term_months": term, "original_balance": (price * (1-deposit) + rng.uniform(25,250,n)).round(2),
        "interest_rate": rng.uniform(0.03,0.42,n).round(5), "borrower_age": rng.integers(18,86,n),
        "annual_income": rng.lognormal(np.log(18000),0.65,n).clip(3000,120000).round(2),
        "credit_score": rng.normal(560,95,n).clip(250,850).round().astype(int), "deposit_pct": deposit.round(5)
    })
    snapshots=[]; prior_dpd=np.zeros(n,dtype=int); prior_default=np.zeros(n,dtype=int)
    for m,date in enumerate(DATES,1):
        d=base.copy(); d["month_no"]=m; d["reporting_date"]=date
        d["months_on_book"]=((date.year-d.loan_start_date.dt.year)*12+date.month-d.loan_start_date.dt.month).clip(lower=0)
        d["remaining_term_months"]=(d.original_term_months-d.months_on_book).clip(lower=1)
        runoff=(d.months_on_book/d.original_term_months).clip(0,0.92)
        d["outstanding_balance"]=(d.original_balance*(1-runoff)*rng.normal(1,0.035,n)).clip(20).round(2)
        latent=-3+0.010*(600-d.credit_score)+2.3*(d.interest_rate-0.12)+0.000018*(22000-d.annual_income)+0.016*prior_dpd+0.22*(m-1)
        arrears=rng.binomial(1,sigmoid(latent)); bands=rng.choice([5,15,35,65,95,150],n,p=[.15,.22,.25,.19,.13,.06])
        d["days_past_due"]=np.where(arrears,bands+rng.integers(-3,4,n),0).clip(0)
        pdef=sigmoid(-5.3+0.042*d.days_past_due+0.009*(520-d.credit_score)+1.2*d.vehicle_condition.eq("Used").astype(float)+0.8*prior_default)
        d["default_status"]=np.maximum(prior_default,rng.binomial(1,pdef)).astype(int)
        d.loc[d.default_status.eq(1),"days_past_due"]=np.maximum(d.loc[d.default_status.eq(1),"days_past_due"],90)
        d["stage"]=np.select([d.default_status.eq(1),d.days_past_due.ge(30)],[3,2],default=1).astype(int)
        d["ead"]=(d.outstanding_balance*(1+0.08*d.stage.ge(2))).round(2)
        d["collateral_value"]=(d.vehicle_price*np.where(d.vehicle_condition.eq("New"),.72,.38)*np.exp(-.015*d.months_on_book)).clip(10).round(2)
        d["loan_to_value"]=(d.ead/d.collateral_value).clip(0,20).round(5)
        rec=sigmoid(1-.32*d.loan_to_value-.012*d.days_past_due+.35*d.vehicle_condition.eq("New").astype(float)+rng.normal(0,.4,n))
        d["observed_recovery_amount"]=np.where(d.default_status.eq(1),d.ead*rec,np.nan).round(2)
        d["observed_lgd"]=np.where(d.default_status.eq(1),1-d.observed_recovery_amount/d.ead,np.nan)
        snapshots.append(d); prior_dpd=d.days_past_due.to_numpy(); prior_default=d.default_status.to_numpy()
    return snapshots


def prep(cat,num):
    return ColumnTransformer([("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",OneHotEncoder(handle_unknown="ignore"))]),cat),
                              ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),num)])


def train(s):
    t=pd.concat(s[:2],ignore_index=True); cat=["car_brand","vehicle_condition"]
    num=["vehicle_price","original_term_months","interest_rate","borrower_age","annual_income","credit_score","deposit_pct","months_on_book","remaining_term_months","outstanding_balance","days_past_due","collateral_value","loan_to_value","ead"]
    f=cat+num
    pdm=Pipeline([("prep",prep(cat,num)),("model",LogisticRegression(max_iter=1500,class_weight="balanced",random_state=SEED))]).fit(t[f],t.default_status)
    td=t[t.default_status.eq(1)]
    lgdm=Pipeline([("prep",prep(cat,num)),("model",HistGradientBoostingRegressor(max_iter=150,learning_rate=.06,max_depth=4,random_state=SEED))]).fit(td[f],td.observed_lgd)
    return pdm,lgdm,f


def apply_shock(m4,m5,brand="Europa"):
    rng=np.random.default_rng(SEED+5); m5=m5.copy(); prior=m4.set_index("account_id").default_status
    prior_d=m5.account_id.map(prior).astype(int); target=m5.car_brand.eq(brand)&m5.vehicle_condition.eq("Used"); eligible=target&prior_d.eq(0)
    baseline=eligible&m5.default_status.eq(1); count=int(baseline.sum()); candidates=m5.index[eligible&m5.default_status.eq(0)].to_numpy()
    selected=rng.choice(candidates,size=min(count,len(candidates)),replace=False)
    m5["month5_default_shock"]=0; m5.loc[selected,"month5_default_shock"]=1; m5.loc[selected,"default_status"]=1
    m5.loc[selected,"days_past_due"]=np.maximum(m5.loc[selected,"days_past_due"],90); m5.loc[selected,"stage"]=3
    m5.loc[selected,"ead"]=(m5.loc[selected,"outstanding_balance"]*1.08).round(2)
    return m5,{"brand":brand,"eligible":int(eligible.sum()),"baseline_new_defaults":count,"additional_defaults":len(selected),"shocked_new_defaults":count+len(selected)}


def score(d,pdm,lgdm,f):
    d=d.copy(); d["modelled_pd"]=pdm.predict_proba(d[f])[:,1].clip(.0001,.9999); d["modelled_lgd"]=lgdm.predict(d[f]).clip(.05,.95)
    mult=np.where(d.stage.eq(1),1,np.minimum(d.remaining_term_months/12,5)); d["ifrs9_pd"]=(d.modelled_pd*mult).clip(.0001,1)
    d.loc[d.stage.eq(3),"ifrs9_pd"]=1; d["ecl"]=(d.ifrs9_pd*d.modelled_lgd*d.ead).round(2); return d


def matrix(m4,m5,mask=None):
    x=m4[["account_id","stage"]].rename(columns={"stage":"month_4"}).merge(m5[["account_id","stage"]].rename(columns={"stage":"month_5"}),on="account_id")
    if mask is not None: x=x[x.account_id.isin(set(m5.loc[mask,"account_id"]))]
    return pd.crosstab(x.month_4,x.month_5,margins=True)


def main(n):
    s=generate_snapshots(n); m4=s[3]; m5,shock=apply_shock(m4,s[4]); pdm,lgdm,f=train(s); m5=score(m5,pdm,lgdm,f)
    target=m5.car_brand.eq("Europa")&m5.vehicle_condition.eq("Used"); overall=matrix(m4,m5); segment=matrix(m4,m5,target)
    summary=pd.DataFrame([{**shock,"accounts":len(m5),"defaults":int(m5.default_status.sum()),"stage_1":int(m5.stage.eq(1).sum()),"stage_2":int(m5.stage.eq(2).sum()),"stage_3":int(m5.stage.eq(3).sum()),"total_ead":m5.ead.sum(),"total_ecl":m5.ecl.sum()}])
    outputs={"synthetic_auto_loans_month_5_scored":m5,"month_4_to_5_stage_transition_overall":overall,"month_4_to_5_stage_transition_used_europa":segment,"synthetic_auto_loans_month_5_summary":summary}
    print(summary.to_string(index=False)); print("\nOverall transition:\n",overall); print("\nUsed Europa transition:\n",segment)
    return outputs


def run(n=6000):
    stop=threading.Event(); th=threading.Thread(target=heartbeat,args=(stop,),daemon=True); th.start()
    try: return main(n)
    finally: stop.set(); th.join(timeout=1)


outputs=run(6000)
synthetic_auto_loans_month_5_scored=outputs["synthetic_auto_loans_month_5_scored"]
month_4_to_5_stage_transition_overall=outputs["month_4_to_5_stage_transition_overall"]
month_4_to_5_stage_transition_used_europa=outputs["month_4_to_5_stage_transition_used_europa"]
synthetic_auto_loans_month_5_summary=outputs["synthetic_auto_loans_month_5_summary"]


In [0]:
# Purpose: A bounded, read-only investigation agent over the monthly IFRS 9 auto-loan
# portfolio. Qwen2.5-1.5B-Instruct receives six native analytics tools, starts from
# portfolio-level results, and decides for itself which views to request. The model is
# given no information about the planted anomaly: no tool exposes the technical
# month5_default_shock indicator, so any deterioration must be inferred from ordinary
# portfolio measures.
# Requires the previous cell to have been executed (its functions and `outputs`).

import json
import re

from transformers import AutoTokenizer, GenerationConfig, pipeline

# 1. Load the model. Qwen2.5-1.5B-Instruct natively renders the `tools` argument in its
#    chat template and replies with its <tool_call {json} </tool_call protocol; this
#    transformers version does not auto-parse it, so step 6 parses it manually.
#    The tokenizer is loaded with clean_up_tokenization_spaces=False (cleanup is
#    destructive for this BPE tokenizer and only produced a warning), and all
#    generation parameters live in a single GenerationConfig object: passing them as
#    pipeline kwargs alongside the model's own generation_config is deprecated, and
#    leaving max_length at its default of 20 collides with max_new_tokens.
_qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", clean_up_tokenization_spaces=False)
qwen_pipeline = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", tokenizer=_qwen_tokenizer, device_map="auto")
AGENT_GENERATION_CONFIG = GenerationConfig(max_new_tokens=1024, do_sample=True, temperature=0.1, max_length=None)

# 2. Rebuild the monthly portfolio in memory (deterministic: the fixed SEED reproduces
#    the previous cell's data exactly). Months 1-4 are scored with the same models
#    trained on months 1-2; month 5 is the shocked, scored frame produced above.
_snapshots = generate_snapshots(6000)
_pdm, _lgdm, _features = train(_snapshots)
portfolio = {m: score(_snapshots[m - 1], _pdm, _lgdm, _features) for m in range(1, 5)}
portfolio[5] = outputs["synthetic_auto_loans_month_5_scored"]

GROUP_KEYS = {
    "brand": ["car_brand"],
    "vehicle_condition": ["vehicle_condition"],
    "brand_and_condition": ["car_brand", "vehicle_condition"],
    "stage": ["stage"],
}

# 3. Read-only tool implementations. Python checks the allow-list, validates every
#    argument inside the tool, and pandas performs the calculation; the JSON result goes
#    back through the native tool role. The model can never execute code, submit pandas
#    expressions, read files, modify the frames, or choose arbitrary column names --
#    group keys, brands and conditions are checked against fixed vocabularies, and
#    month5_default_shock is not exposed by any tool.

def _check_month(value, label):
    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 5:
        raise ValueError(f"{label} must be an integer month number from 1 to 5; received {value!r}")
    return value

def _check_group_by(value):
    if value not in GROUP_KEYS:
        raise ValueError(f"group_by must be one of {sorted(GROUP_KEYS)}; received {value!r}")
    return value

def _segment_series(df, group_by):
    keys = GROUP_KEYS[group_by]
    if len(keys) == 2:
        return df[keys[0]].astype(str) + " / " + df[keys[1]].astype(str)
    return df[keys[0]].astype(str)

def _records(df):
    # to_json normalises numpy scalars and NaN -> null, keeping results JSON-serializable
    return json.loads(df.to_json(orient="records"))

def get_available_data():
    return {
        "description": "Monthly snapshots of a synthetic auto-loan portfolio scored under IFRS 9. The same accounts are tracked across all months.",
        "months": [{"month": m, "reporting_date": str(portfolio[m].reporting_date.iloc[0].date()), "accounts": int(len(portfolio[m]))} for m in sorted(portfolio)],
        "permitted_analyses": [
            "Portfolio summary of one month: accounts, default rate, stage distribution, EAD, ECL, modelled PD, modelled LGD and arrears",
            "Comparison of two months by brand, vehicle condition, brand and condition, or stage",
            "Attribution of the ECL movement between two months to those segments",
            "Stage transition matrix between two months, optionally filtered by brand or vehicle condition",
            "New defaults between two months among accounts not already in default at the start",
        ],
    }

def get_portfolio_summary(month):
    _check_month(month, "month")
    d = portfolio[month]
    return {
        "month": month,
        "reporting_date": str(d.reporting_date.iloc[0].date()),
        "accounts": int(len(d)),
        "default_rate": round(float(d.default_status.mean()), 4),
        "stage_accounts": {f"stage_{s}": int((d.stage == s).sum()) for s in (1, 2, 3)},
        "total_ead": round(float(d.ead.sum()), 2),
        "total_ecl": round(float(d.ecl.sum()), 2),
        "avg_modelled_pd": round(float(d.modelled_pd.mean()), 4),
        "avg_modelled_lgd": round(float(d.modelled_lgd.mean()), 4),
        "avg_ifrs9_pd": round(float(d.ifrs9_pd.mean()), 4),
        "arrears": {
            "pct_accounts_in_arrears": round(float((d.days_past_due > 0).mean()), 4),
            "avg_days_past_due": round(float(d.days_past_due.mean()), 2),
            "pct_dpd_30_plus": round(float((d.days_past_due >= 30).mean()), 4),
        },
    }

def _segment_metrics(df, group_by):
    g = df.groupby(_segment_series(df, group_by))
    return pd.DataFrame({
        "accounts": g.size(),
        "default_rate": g.default_status.mean(),
        "total_ead": g.ead.sum(),
        "total_ecl": g.ecl.sum(),
        "avg_modelled_pd": g.modelled_pd.mean(),
        "avg_modelled_lgd": g.modelled_lgd.mean(),
    })

def compare_months(start_month, end_month, group_by):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)
    a = _segment_metrics(portfolio[start_month], group_by).add_suffix("_start")
    b = _segment_metrics(portfolio[end_month], group_by).add_suffix("_end")
    j = a.join(b, how="outer").fillna(0.0)
    j["accounts_start"], j["accounts_end"] = j.accounts_start.astype(int), j.accounts_end.astype(int)
    j["ecl_change"] = j.total_ecl_end - j.total_ecl_start
    j["ecl_change_pct"] = j.ecl_change / j.total_ecl_start.replace(0, pd.NA) * 100
    return {"start_month": start_month, "end_month": end_month, "group_by": group_by,
            "segments": _records(j.rename_axis("segment").reset_index())}

def get_ecl_movement_by_segment(start_month, end_month):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    total_change = float(portfolio[end_month].ecl.sum() - portfolio[start_month].ecl.sum())
    breakdowns = {}
    for gb in ("brand", "vehicle_condition", "brand_and_condition", "stage"):
        a = portfolio[start_month].groupby(_segment_series(portfolio[start_month], gb)).ecl.sum().rename("ecl_start")
        b = portfolio[end_month].groupby(_segment_series(portfolio[end_month], gb)).ecl.sum().rename("ecl_end")
        j = pd.concat([a, b], axis=1).fillna(0.0)
        j["ecl_change"] = j.ecl_end - j.ecl_start
        j["ecl_change_pct"] = j.ecl_change / j.ecl_start.replace(0, pd.NA) * 100
        j["share_of_total_ecl_change_pct"] = j.ecl_change / total_change * 100 if total_change != 0 else pd.NA
        breakdowns[gb] = _records(j.rename_axis("segment").reset_index())
    return {"start_month": start_month, "end_month": end_month,
            "total_ecl_change": round(total_change, 2), "breakdowns": breakdowns}

def get_stage_transition_matrix(start_month, end_month, brand=None, vehicle_condition=None):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    a, b, filters = portfolio[start_month], portfolio[end_month], {}
    if brand is not None:
        if brand not in BRANDS:
            raise ValueError(f"brand must be one of {BRANDS}; received {brand!r}")
        filters["brand"] = brand
        a, b = a[a.car_brand == brand], b[b.car_brand == brand]
    if vehicle_condition is not None:
        if vehicle_condition not in ("New", "Used"):
            raise ValueError(f"vehicle_condition must be 'New' or 'Used'; received {vehicle_condition!r}")
        filters["vehicle_condition"] = vehicle_condition
        a, b = a[a.vehicle_condition == vehicle_condition], b[b.vehicle_condition == vehicle_condition]
    x = a[["account_id", "stage"]].rename(columns={"stage": "from_stage"}).merge(
        b[["account_id", "stage"]].rename(columns={"stage": "to_stage"}), on="account_id")
    ct = pd.crosstab(x.from_stage, x.to_stage)
    matrix = {str(fs): {str(ts): int(n) for ts, n in row.items()} for fs, row in ct.iterrows()}
    return {"start_month": start_month, "end_month": end_month, "filters": filters,
            "accounts": int(len(x)), "transition_matrix": matrix}

def get_new_default_analysis(start_month, end_month, group_by="brand_and_condition"):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)
    a, b = portfolio[start_month], portfolio[end_month]
    x = a.loc[a.default_status == 0, ["account_id"]].merge(b[["account_id", "default_status"]], on="account_id")
    x["segment"] = x.account_id.map(dict(zip(a.account_id, _segment_series(a, group_by))))
    by_segment = x.groupby("segment").agg(accounts=("account_id", "size"),
                                          new_defaults=("default_status", "sum"),
                                          new_default_rate=("default_status", "mean")).reset_index()
    by_segment["new_default_rate"] = by_segment.new_default_rate.round(4)
    return {"start_month": start_month, "end_month": end_month, "group_by": group_by,
            "overall": {"accounts_not_defaulted_at_start": int(len(x)),
                        "new_defaults": int(x.default_status.sum()),
                        "new_default_rate": round(float(x.default_status.mean()), 4)},
            "by_segment": _records(by_segment)}

TOOL_REGISTRY = {
    "get_available_data": get_available_data,
    "get_portfolio_summary": get_portfolio_summary,
    "compare_months": compare_months,
    "get_ecl_movement_by_segment": get_ecl_movement_by_segment,
    "get_stage_transition_matrix": get_stage_transition_matrix,
    "get_new_default_analysis": get_new_default_analysis,
}

# 4. Tool schemas handed to the model. Explicit JSON schemas, so the vocabulary the
#    model sees is exactly the vocabulary the tools validate against.
TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "get_available_data",
        "description": "Describe the available monthly portfolio data: the months, their reporting dates and account counts, and the permitted analyses.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {
        "name": "get_portfolio_summary",
        "description": "Return portfolio-level credit-risk metrics for one month: accounts, default rate, stage distribution, EAD, ECL, modelled PD, modelled LGD and arrears measures.",
        "parameters": {"type": "object", "properties": {
            "month": {"type": "integer", "description": "Month number to summarise (1 to 5)."}},
            "required": ["month"]}}},
    {"type": "function", "function": {
        "name": "compare_months",
        "description": "Compare two months segment by segment. For each segment, returns accounts, default rate, EAD, ECL, modelled PD and modelled LGD in both months, plus the change in ECL.",
        "parameters": {"type": "object", "properties": {
            "start_month": {"type": "integer", "description": "Earlier month number (1 to 5)."},
            "end_month": {"type": "integer", "description": "Later month number (1 to 5)."},
            "group_by": {"type": "string", "enum": ["brand", "vehicle_condition", "brand_and_condition", "stage"],
                          "description": "Segmentation for the comparison."}},
            "required": ["start_month", "end_month", "group_by"]}}},
    {"type": "function", "function": {
        "name": "get_ecl_movement_by_segment",
        "description": "Attribute the ECL movement between two months to segments: by brand, by vehicle condition, by brand and condition, and by stage. Each segment shows ECL in both months, the change, and its share of the total movement.",
        "parameters": {"type": "object", "properties": {
            "start_month": {"type": "integer", "description": "Earlier month number (1 to 5)."},
            "end_month": {"type": "integer", "description": "Later month number (1 to 5)."}},
            "required": ["start_month", "end_month"]}}},
    {"type": "function", "function": {
        "name": "get_stage_transition_matrix",
        "description": "Calculate the IFRS 9 stage transition matrix between two months, with optional filters on brand or vehicle condition.",
        "parameters": {"type": "object", "properties": {
            "start_month": {"type": "integer", "description": "Earlier month number (1 to 5)."},
            "end_month": {"type": "integer", "description": "Later month number (1 to 5)."},
            "brand": {"type": "string", "enum": ["Io", "Europa", "Ganymede", "Callisto", "Amalthea"], "description": "Optional brand filter."},
            "vehicle_condition": {"type": "string", "enum": ["New", "Used"], "description": "Optional vehicle-condition filter."}},
            "required": ["start_month", "end_month"]}}},
    {"type": "function", "function": {
        "name": "get_new_default_analysis",
        "description": "Measure new defaults between two months among accounts that were not already in default at the start of the comparison, overall and by segment.",
        "parameters": {"type": "object", "properties": {
            "start_month": {"type": "integer", "description": "Earlier month number (1 to 5)."},
            "end_month": {"type": "integer", "description": "Later month number (1 to 5)."},
            "group_by": {"type": "string", "enum": ["brand", "vehicle_condition", "brand_and_condition", "stage"],
                          "description": "Optional segmentation; defaults to brand_and_condition."}},
            "required": ["start_month", "end_month"]}}},
]

# 5. Prompts. Deliberately neutral: no mention of Europa, used cars, month 5 or any
#    planted shock. The system prompt only sets investigative discipline.
SYSTEM_PROMPT = (
    "You are a credit-risk analyst reviewing monthly IFRS 9 auto-loan portfolio results. "
    "Begin with broad, portfolio-level checks before drilling down. "
    "Drill down into segments only when the evidence you have already seen supports it. "
    "Do not assume a cause for any movement; keep candidate explanations separate as hypotheses. "
    "Quantify material movements with numbers, in absolute and percentage terms. "
    "Clearly distinguish findings (supported by tool results you retrieved) from hypotheses (plausible but not yet verified). "
    "Stop investigating once you have enough evidence, and then answer with an ordinary response summarising the findings, the supporting figures, and any remaining hypotheses. "
    "Always use the provided tools to obtain data; never guess values. Never state a figure you did not retrieve from a tool result; if a statement needs data you have not retrieved, call a tool first. "
    "Compare at least the earliest and the latest month before concluding whether the results are stable."
)

USER_PROMPT = (
    "Review the available monthly IFRS 9 auto-loan portfolio data. Determine whether the "
    "portfolio and credit-risk results are stable over time. If they are not stable, "
    "investigate the data and explain the most important findings you can support."
)

# 6. The bounded investigation loop. Up to 10 rounds: Qwen requests a tool, Python
#    checks the allow-list and validates the arguments, pandas computes the result, and
#    the JSON goes back through the native tool role. Every requested tool, argument and
#    returned result is printed. An ordinary response (no tool call) ends the loop.
TOOL_CALL_PATTERN = re.compile(r"<tool_call\s*>?\s*(\{.*?\})\s*</tool_call\s*>?", re.DOTALL)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]
print(f"User prompt: {USER_PROMPT}\n")

MAX_ROUNDS = 10
used_tools = set()          # analytical tools the agent has actually retrieved data through
retrieval_pushbacks = 0      # final answers rejected for lacking any retrieved metrics
for round_no in range(1, MAX_ROUNDS + 1):
    response = qwen_pipeline(messages, tools=TOOL_SCHEMAS, generation_config=AGENT_GENERATION_CONFIG)
    assistant_content = response[0]["generated_text"][-1]["content"] or ""
    messages.append({"role": "assistant", "content": assistant_content})
    tool_matches = list(TOOL_CALL_PATTERN.finditer(assistant_content))
    if not tool_matches:
        # Anti-hallucination guard: a 1.5B model sometimes concludes immediately,
        # inventing segment figures it never retrieved. A final answer is accepted
        # only after at least one analytical tool (not just the data inventory) has
        # been used; otherwise the agent is sent back to the tools.
        if used_tools - {"get_available_data"}:
            print("=" * 80)
            print(f"FINAL ANSWER after {round_no} round(s), no further tool requested:")
            print(assistant_content)
            break
        retrieval_pushbacks += 1
        print(f"[round {round_no}] final answer rejected: no analytical data retrieved yet; sending the agent back to the tools.\n")
        messages.append({"role": "user", "content": "You have only described the available data. Retrieve portfolio metrics with the tools (for example, monthly portfolio summaries and month-to-month comparisons) before drawing any conclusion, and never state figures you have not retrieved."})
        continue
    for match in tool_matches:
        name, result = "unknown", None
        try:
            request = json.loads(match.group(1))
            name = request.get("name", "unknown")
            arguments = request.get("arguments", {})
            if name not in TOOL_REGISTRY:
                result = {"error": f"Tool is not permitted: {name!r}. Allowed tools: {sorted(TOOL_REGISTRY)}"}
            else:
                print(f"[round {round_no}] {name} requested with arguments: {json.dumps(arguments)}")
                result = TOOL_REGISTRY[name](**arguments)
                used_tools.add(name)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
        payload = json.dumps(result)
        print(f"[round {round_no}] {name} result: {payload}\n")
        messages.append({"role": "tool", "name": name, "content": payload})
else:
    print("=" * 80)
    print(f"Round limit of {MAX_ROUNDS} reached without a final answer ({retrieval_pushbacks} rejected for missing retrieval); stopping the investigation.")

In [0]:
# Purpose: Run a four-phase, evidence-gated IFRS 9 portfolio investigation with Qwen2.5-1.5B-Instruct and read-only Pandas tools.
# Prerequisite: Run the previous data-generation cell first so generate_snapshots, train, score, outputs, BRANDS and pd are available.

import json
import re
import threading
from copy import deepcopy

from transformers import AutoTokenizer, GenerationConfig, pipeline


# -----------------------------------------------------------------------------
# 1. Model loading and deterministic generation
# -----------------------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

_qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    clean_up_tokenization_spaces=False,
)

qwen_pipeline = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=_qwen_tokenizer,
    device_map="auto",
)

AGENT_GENERATION_CONFIG = GenerationConfig(
    max_new_tokens=1024,
    do_sample=False,
    max_length=None,
)


# -----------------------------------------------------------------------------
# 2. Rebuild the deterministic in-memory portfolio
# -----------------------------------------------------------------------------

_snapshots = generate_snapshots(6000)
_pdm, _lgdm, _features = train(_snapshots)

portfolio = {
    month: score(_snapshots[month - 1], _pdm, _lgdm, _features)
    for month in range(1, 5)
}
portfolio[5] = outputs["synthetic_auto_loans_month_5_scored"]

GROUP_KEYS = {
    "brand": ["car_brand"],
    "vehicle_condition": ["vehicle_condition"],
    "brand_and_condition": ["car_brand", "vehicle_condition"],
    "stage": ["stage"],
}


# -----------------------------------------------------------------------------
# 3. Validation and read-only Pandas tools
# -----------------------------------------------------------------------------

def _check_month(value, label):
    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 5:
        raise ValueError(
            f"{label} must be an integer month number from 1 to 5; received {value!r}"
        )
    return value


def _check_group_by(value):
    if value not in GROUP_KEYS:
        raise ValueError(
            f"group_by must be one of {sorted(GROUP_KEYS)}; received {value!r}"
        )
    return value


def _segment_series(df, group_by):
    keys = GROUP_KEYS[group_by]
    if len(keys) == 2:
        return df[keys[0]].astype(str) + " / " + df[keys[1]].astype(str)
    return df[keys[0]].astype(str)


def _records(df):
    return json.loads(df.to_json(orient="records"))


def _percentage_change(start_value, end_value):
    if start_value == 0:
        return None
    return round((end_value / start_value - 1.0) * 100.0, 2)


def get_available_data():
    return {
        "description": (
            "Monthly snapshots of a synthetic auto-loan portfolio scored under "
            "IFRS 9. The same accounts are tracked across all months."
        ),
        "months": [
            {
                "month": month,
                "reporting_date": str(portfolio[month].reporting_date.iloc[0].date()),
                "accounts": int(len(portfolio[month])),
            }
            for month in sorted(portfolio)
        ],
        "permitted_analyses": [
            "Portfolio summary and movement between any two months",
            "Comparison by brand, vehicle condition, brand and condition, or stage",
            "ECL movement attribution by segment",
            "Stage transition matrix with optional brand and condition filters",
            "New-default analysis among accounts not already defaulted at the start",
        ],
    }


def get_portfolio_summary(month):
    _check_month(month, "month")
    d = portfolio[month]
    default_rate = float(d.default_status.mean())
    avg_pd = float(d.modelled_pd.mean())
    avg_lgd = float(d.modelled_lgd.mean())
    avg_ifrs9_pd = float(d.ifrs9_pd.mean())
    return {
        "month": month,
        "reporting_date": str(d.reporting_date.iloc[0].date()),
        "accounts": int(len(d)),
        "defaults": int(d.default_status.sum()),
        "default_rate_decimal": round(default_rate, 4),
        "default_rate_pct": round(default_rate * 100, 2),
        "stage_accounts": {
            f"stage_{stage}": int((d.stage == stage).sum()) for stage in (1, 2, 3)
        },
        "total_ead": round(float(d.ead.sum()), 2),
        "total_ead_millions": round(float(d.ead.sum()) / 1_000_000, 4),
        "total_ecl": round(float(d.ecl.sum()), 2),
        "total_ecl_thousands": round(float(d.ecl.sum()) / 1_000, 2),
        "avg_modelled_pd_decimal": round(avg_pd, 4),
        "avg_modelled_pd_pct": round(avg_pd * 100, 2),
        "avg_modelled_lgd_decimal": round(avg_lgd, 4),
        "avg_modelled_lgd_pct": round(avg_lgd * 100, 2),
        "avg_ifrs9_pd_decimal": round(avg_ifrs9_pd, 4),
        "avg_ifrs9_pd_pct": round(avg_ifrs9_pd * 100, 2),
        "arrears": {
            "pct_accounts_in_arrears": round(float((d.days_past_due > 0).mean()) * 100, 2),
            "avg_days_past_due": round(float(d.days_past_due.mean()), 2),
            "pct_dpd_30_plus": round(float((d.days_past_due >= 30).mean()) * 100, 2),
        },
    }


def compare_portfolio_totals(start_month, end_month):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    start = portfolio[start_month]
    end = portfolio[end_month]

    start_ead = float(start.ead.sum())
    end_ead = float(end.ead.sum())
    start_ecl = float(start.ecl.sum())
    end_ecl = float(end.ecl.sum())
    start_default_rate = float(start.default_status.mean())
    end_default_rate = float(end.default_status.mean())

    return {
        "start_month": start_month,
        "end_month": end_month,
        "ead": {
            "start": round(start_ead, 2),
            "end": round(end_ead, 2),
            "absolute_change": round(end_ead - start_ead, 2),
            "percentage_change": _percentage_change(start_ead, end_ead),
        },
        "ecl": {
            "start": round(start_ecl, 2),
            "end": round(end_ecl, 2),
            "absolute_change": round(end_ecl - start_ecl, 2),
            "percentage_change": _percentage_change(start_ecl, end_ecl),
        },
        "defaults": {
            "start_count": int(start.default_status.sum()),
            "end_count": int(end.default_status.sum()),
            "absolute_change": int(end.default_status.sum() - start.default_status.sum()),
            "start_rate_pct": round(start_default_rate * 100, 2),
            "end_rate_pct": round(end_default_rate * 100, 2),
            "rate_change_percentage_points": round(
                (end_default_rate - start_default_rate) * 100, 2
            ),
        },
        "stage_3": {
            "start_count": int(start.stage.eq(3).sum()),
            "end_count": int(end.stage.eq(3).sum()),
            "absolute_change": int(end.stage.eq(3).sum() - start.stage.eq(3).sum()),
        },
    }


def _segment_metrics(df, group_by):
    grouped = df.groupby(_segment_series(df, group_by))
    result = pd.DataFrame({
        "accounts": grouped.size(),
        "defaults": grouped.default_status.sum(),
        "default_rate_pct": grouped.default_status.mean() * 100,
        "total_ead": grouped.ead.sum(),
        "total_ecl": grouped.ecl.sum(),
        "avg_modelled_pd_pct": grouped.modelled_pd.mean() * 100,
        "avg_modelled_lgd_pct": grouped.modelled_lgd.mean() * 100,
    })
    return result


def compare_months(start_month, end_month, group_by):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)

    start = _segment_metrics(portfolio[start_month], group_by).add_suffix("_start")
    end = _segment_metrics(portfolio[end_month], group_by).add_suffix("_end")
    joined = start.join(end, how="outer").fillna(0.0)

    joined["accounts_start"] = joined.accounts_start.astype(int)
    joined["accounts_end"] = joined.accounts_end.astype(int)
    joined["defaults_start"] = joined.defaults_start.astype(int)
    joined["defaults_end"] = joined.defaults_end.astype(int)
    joined["default_rate_change_pp"] = (
        joined.default_rate_pct_end - joined.default_rate_pct_start
    )
    joined["ecl_change"] = joined.total_ecl_end - joined.total_ecl_start
    joined["ecl_change_pct"] = (
        joined.ecl_change / joined.total_ecl_start.replace(0, pd.NA) * 100
    )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "group_by": group_by,
        "segments": _records(joined.rename_axis("segment").reset_index()),
    }


def get_ecl_movement_by_segment(start_month, end_month):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")

    total_change = float(
        portfolio[end_month].ecl.sum() - portfolio[start_month].ecl.sum()
    )
    breakdowns = {}

    for group_by in ("brand", "vehicle_condition", "brand_and_condition", "stage"):
        start = (
            portfolio[start_month]
            .groupby(_segment_series(portfolio[start_month], group_by))
            .ecl.sum()
            .rename("ecl_start")
        )
        end = (
            portfolio[end_month]
            .groupby(_segment_series(portfolio[end_month], group_by))
            .ecl.sum()
            .rename("ecl_end")
        )
        joined = pd.concat([start, end], axis=1).fillna(0.0)
        joined["ecl_change"] = joined.ecl_end - joined.ecl_start
        joined["ecl_change_pct"] = (
            joined.ecl_change / joined.ecl_start.replace(0, pd.NA) * 100
        )
        joined["share_of_total_ecl_change_pct"] = (
            joined.ecl_change / total_change * 100 if total_change != 0 else pd.NA
        )
        joined = joined.sort_values("ecl_change", ascending=False)
        breakdowns[group_by] = _records(
            joined.rename_axis("segment").reset_index()
        )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "total_ecl_change": round(total_change, 2),
        "breakdowns": breakdowns,
    }


def get_stage_transition_matrix(
    start_month,
    end_month,
    brand=None,
    vehicle_condition=None,
):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")

    start = portfolio[start_month]
    end = portfolio[end_month]
    filters = {}

    if brand is not None:
        if brand not in BRANDS:
            raise ValueError(f"brand must be one of {BRANDS}; received {brand!r}")
        filters["brand"] = brand
        start = start[start.car_brand == brand]
        end = end[end.car_brand == brand]

    if vehicle_condition is not None:
        if vehicle_condition not in ("New", "Used"):
            raise ValueError(
                "vehicle_condition must be 'New' or 'Used'; "
                f"received {vehicle_condition!r}"
            )
        filters["vehicle_condition"] = vehicle_condition
        start = start[start.vehicle_condition == vehicle_condition]
        end = end[end.vehicle_condition == vehicle_condition]

    transitions = (
        start[["account_id", "stage"]]
        .rename(columns={"stage": "from_stage"})
        .merge(
            end[["account_id", "stage"]].rename(columns={"stage": "to_stage"}),
            on="account_id",
        )
    )
    counts = (
        transitions.groupby(["from_stage", "to_stage"])
        .size()
        .rename("accounts")
        .reset_index()
    )
    counts["from_stage_total"] = counts.groupby("from_stage").accounts.transform("sum")
    counts["row_percentage"] = counts.accounts / counts.from_stage_total * 100

    return {
        "start_month": start_month,
        "end_month": end_month,
        "filters": filters,
        "accounts": int(len(transitions)),
        "transitions": _records(counts),
    }


def get_new_default_analysis(
    start_month,
    end_month,
    group_by="brand_and_condition",
):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)

    start = portfolio[start_month]
    end = portfolio[end_month]
    joined = (
        start.loc[start.default_status == 0, ["account_id"]]
        .merge(end[["account_id", "default_status", "ecl"]], on="account_id")
    )
    joined["segment"] = joined.account_id.map(
        dict(zip(start.account_id, _segment_series(start, group_by)))
    )

    by_segment = (
        joined.groupby("segment")
        .agg(
            eligible_accounts=("account_id", "size"),
            new_defaults=("default_status", "sum"),
            end_ecl=("ecl", "sum"),
        )
        .reset_index()
    )
    by_segment["new_default_rate_pct"] = (
        by_segment.new_defaults / by_segment.eligible_accounts * 100
    )
    by_segment = by_segment.sort_values(
        ["new_default_rate_pct", "new_defaults"],
        ascending=False,
    )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "group_by": group_by,
        "overall": {
            "accounts_not_defaulted_at_start": int(len(joined)),
            "new_defaults": int(joined.default_status.sum()),
            "new_default_rate_pct": round(float(joined.default_status.mean()) * 100, 2),
        },
        "by_segment": _records(by_segment),
    }


TOOL_REGISTRY = {
    "get_available_data": get_available_data,
    "get_portfolio_summary": get_portfolio_summary,
    "compare_portfolio_totals": compare_portfolio_totals,
    "compare_months": compare_months,
    "get_ecl_movement_by_segment": get_ecl_movement_by_segment,
    "get_stage_transition_matrix": get_stage_transition_matrix,
    "get_new_default_analysis": get_new_default_analysis,
}


# -----------------------------------------------------------------------------
# 4. Explicit tool schemas and phase-specific tool access
# -----------------------------------------------------------------------------

ALL_TOOL_SCHEMAS = {
    "get_available_data": {
        "type": "function",
        "function": {
            "name": "get_available_data",
            "description": "Describe available monthly portfolio data and analyses.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    "get_portfolio_summary": {
        "type": "function",
        "function": {
            "name": "get_portfolio_summary",
            "description": "Return portfolio-level risk and ECL metrics for one month.",
            "parameters": {
                "type": "object",
                "properties": {
                    "month": {"type": "integer", "minimum": 1, "maximum": 5}
                },
                "required": ["month"],
            },
        },
    },
    "compare_portfolio_totals": {
        "type": "function",
        "function": {
            "name": "compare_portfolio_totals",
            "description": "Calculate exact portfolio-level movements between two months.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "end_month": {"type": "integer", "minimum": 1, "maximum": 5},
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    "compare_months": {
        "type": "function",
        "function": {
            "name": "compare_months",
            "description": "Compare two months by an approved segment.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "end_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "group_by": {
                        "type": "string",
                        "enum": ["brand", "vehicle_condition", "brand_and_condition", "stage"],
                    },
                },
                "required": ["start_month", "end_month", "group_by"],
            },
        },
    },
    "get_ecl_movement_by_segment": {
        "type": "function",
        "function": {
            "name": "get_ecl_movement_by_segment",
            "description": "Attribute ECL movement to brands, conditions, combinations and stages.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "end_month": {"type": "integer", "minimum": 1, "maximum": 5},
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    "get_stage_transition_matrix": {
        "type": "function",
        "function": {
            "name": "get_stage_transition_matrix",
            "description": "Calculate stage transitions with optional brand and condition filters.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "end_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "brand": {"type": "string", "enum": BRANDS},
                    "vehicle_condition": {"type": "string", "enum": ["New", "Used"]},
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    "get_new_default_analysis": {
        "type": "function",
        "function": {
            "name": "get_new_default_analysis",
            "description": "Analyse new defaults among accounts not defaulted at the start.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "end_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "group_by": {
                        "type": "string",
                        "enum": ["brand", "vehicle_condition", "brand_and_condition", "stage"],
                    },
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
}

PHASE_TOOL_NAMES = {
    "phase_1_portfolio_stability": [
        "get_available_data",
        "get_portfolio_summary",
        "compare_portfolio_totals",
    ],
    "phase_2_segment_screening": [
        "compare_months",
        "get_ecl_movement_by_segment",
    ],
    "phase_3_mechanism_investigation": [
        "get_stage_transition_matrix",
        "get_new_default_analysis",
    ],
    "phase_4_final_synthesis": [],
}

TOOL_CALL_PATTERN = re.compile(
    r"<tool_call\s*>?\s*(\{.*?\})\s*</tool_call\s*>?",
    re.DOTALL,
)


# -----------------------------------------------------------------------------
# 5. Orchestration helpers
# -----------------------------------------------------------------------------

def _parse_tool_calls(content):
    return [json.loads(match.group(1)) for match in TOOL_CALL_PATTERN.finditer(content)]


def _execute_tool(request, allowed_tool_names):
    name = request.get("name", "unknown")
    arguments = request.get("arguments", {})
    if isinstance(arguments, str):
        arguments = json.loads(arguments)

    if name not in allowed_tool_names:
        return name, {
            "error": f"Tool {name!r} is unavailable in this phase.",
            "available_tools": sorted(allowed_tool_names),
        }
    if name not in TOOL_REGISTRY:
        return name, {"error": f"Tool {name!r} is not registered."}

    try:
        result = TOOL_REGISTRY[name](**arguments)
    except Exception as error:
        result = {"error": f"{type(error).__name__}: {error}"}
    return name, result


def _run_tool_phase(
    phase_name,
    phase_prompt,
    allowed_tool_names,
    required_tool_names,
    evidence_log,
    maximum_rounds=6,
):
    """Run one bounded phase and reject completion until its evidence gate is met."""
    phase_tools = [ALL_TOOL_SCHEMAS[name] for name in allowed_tool_names]
    messages = [
        {
            "role": "system",
            "content": (
                "You are a cautious credit-risk analyst. Use only retrieved evidence. "
                "Never invent, rescale or manually calculate a figure. The tools return "
                "both decimal and percentage values where relevant. Prefer the supplied "
                "percentage values. Complete only the current investigation phase."
            ),
        },
        {"role": "user", "content": phase_prompt},
    ]

    used_in_phase = set()
    final_text = ""

    for round_no in range(1, maximum_rounds + 1):
        response = qwen_pipeline(
            messages,
            tools=phase_tools,
            generation_config=AGENT_GENERATION_CONFIG,
        )
        assistant_content = response[0]["generated_text"][-1]["content"] or ""
        messages.append({"role": "assistant", "content": assistant_content})
        tool_calls = _parse_tool_calls(assistant_content)

        if not tool_calls:
            missing = set(required_tool_names) - used_in_phase
            if missing:
                print(
                    f"[{phase_name}, round {round_no}] phase answer rejected; "
                    f"missing required evidence from: {sorted(missing)}"
                )
                messages.append({
                    "role": "user",
                    "content": (
                        "This phase is incomplete. Retrieve the missing evidence using: "
                        f"{sorted(missing)}. Do not provide the phase conclusion yet."
                    ),
                })
                continue

            final_text = assistant_content
            print(f"\n[{phase_name}] PHASE CONCLUSION:\n{final_text}\n")
            return {
                "phase": phase_name,
                "conclusion": final_text,
                "used_tools": sorted(used_in_phase),
                "evidence": deepcopy(evidence_log[phase_name]),
            }

        for request in tool_calls:
            name, result = _execute_tool(request, set(allowed_tool_names))
            used_in_phase.add(name)
            evidence_item = {
                "tool": name,
                "arguments": request.get("arguments", {}),
                "result": result,
            }
            evidence_log[phase_name].append(evidence_item)
            payload = json.dumps(result)
            print(
                f"[{phase_name}, round {round_no}] {name}"
                f"({json.dumps(request.get('arguments', {}))})\n{payload}\n"
            )
            messages.append({"role": "tool", "name": name, "content": payload})

    raise RuntimeError(
        f"{phase_name} reached its limit of {maximum_rounds} rounds without completion."
    )


def _compact_evidence(evidence_log):
    """Create the final evidence pack without exposing internal shock indicators."""
    return {
        phase: items
        for phase, items in evidence_log.items()
        if phase != "phase_4_final_synthesis"
    }


def _run_final_synthesis(phase_results, evidence_log):
    """Generate an evidence-only final report with no tools available."""
    evidence_pack = _compact_evidence(evidence_log)
    prompt = {
        "task": (
            "Produce the final IFRS 9 portfolio investigation report using only the "
            "verified evidence below. Do not add metrics, abbreviations or explanations "
            "that are absent from the evidence. Do not redo arithmetic."
        ),
        "required_sections": [
            "Overall stability conclusion",
            "Portfolio-level movement",
            "Principal deteriorating segments",
            "Observed deterioration mechanism",
            "Limitations and unverified hypotheses",
        ],
        "phase_conclusions": phase_results,
        "verified_tool_evidence": evidence_pack,
    }

    messages = [
        {
            "role": "system",
            "content": (
                "You are writing an evidence-controlled credit-risk investigation. "
                "Use only the supplied evidence. Copy figures exactly as supplied. "
                "Treat decimal rates and percentage rates as different units. "
                "If evidence does not prove an external cause, say so explicitly."
            ),
        },
        {"role": "user", "content": json.dumps(prompt)},
    ]

    response = qwen_pipeline(
        messages,
        generation_config=AGENT_GENERATION_CONFIG,
    )
    final_answer = response[0]["generated_text"][-1]["content"] or ""
    print("=" * 80)
    print("FINAL FOUR-PHASE INVESTIGATION REPORT")
    print(final_answer)
    return final_answer


# -----------------------------------------------------------------------------
# 6. Four-phase orchestrator
# -----------------------------------------------------------------------------

def run_four_phase_investigation():
    evidence_log = {
        phase: [] for phase in PHASE_TOOL_NAMES
    }
    phase_results = []

    phase_1 = _run_tool_phase(
        phase_name="phase_1_portfolio_stability",
        phase_prompt=(
            "Assess whether the portfolio is stable over the available months. "
            "First confirm the available data. Retrieve all monthly portfolio summaries "
            "and compare the earliest and latest months. Identify which period or periods "
            "warrant deeper investigation. Do not investigate segments yet."
        ),
        allowed_tool_names=PHASE_TOOL_NAMES["phase_1_portfolio_stability"],
        required_tool_names={
            "get_available_data",
            "get_portfolio_summary",
            "compare_portfolio_totals",
        },
        evidence_log=evidence_log,
        maximum_rounds=6,
    )
    phase_results.append(phase_1)

    phase_2 = _run_tool_phase(
        phase_name="phase_2_segment_screening",
        phase_prompt=(
            "The portfolio-level review found instability. Screen the relevant comparison "
            "period for observable segment drivers. Compare brands, vehicle conditions, "
            "brand-condition combinations and stages. Attribute the ECL movement and "
            "identify candidate segments for deeper investigation. Do not claim a root "
            "cause yet. The previous phase conclusion was: "
            + phase_1["conclusion"]
        ),
        allowed_tool_names=PHASE_TOOL_NAMES["phase_2_segment_screening"],
        required_tool_names={"compare_months", "get_ecl_movement_by_segment"},
        evidence_log=evidence_log,
        maximum_rounds=6,
    )
    phase_results.append(phase_2)

    phase_3 = _run_tool_phase(
        phase_name="phase_3_mechanism_investigation",
        phase_prompt=(
            "Investigate the candidate segments identified in the screening phase. Use "
            "stage-transition and new-default evidence to establish the observed mechanism. "
            "Distinguish accounts newly entering default from those already defaulted. "
            "Do not infer an external business cause that is not present in the data. "
            "The screening conclusion was: "
            + phase_2["conclusion"]
        ),
        allowed_tool_names=PHASE_TOOL_NAMES["phase_3_mechanism_investigation"],
        required_tool_names={"get_stage_transition_matrix", "get_new_default_analysis"},
        evidence_log=evidence_log,
        maximum_rounds=6,
    )
    phase_results.append(phase_3)

    final_answer = _run_final_synthesis(phase_results, evidence_log)

    return {
        "phase_results": phase_results,
        "evidence_log": evidence_log,
        "final_answer": final_answer,
    }


# -----------------------------------------------------------------------------
# 7. Execute with a heartbeat
# -----------------------------------------------------------------------------

def _heartbeat(stop_event):
    elapsed = 0
    while not stop_event.wait(5):
        elapsed += 5
        print(
            f"[heartbeat] {elapsed}s elapsed - four-phase investigation is running",
            flush=True,
        )


_stop_event = threading.Event()
_heartbeat_thread = threading.Thread(
    target=_heartbeat,
    args=(_stop_event,),
    daemon=True,
)
_heartbeat_thread.start()

try:
    investigation_output = run_four_phase_investigation()
finally:
    _stop_event.set()
    _heartbeat_thread.join(timeout=1)


In [0]:
# Purpose: A bounded, read-only investigation agent over the monthly IFRS 9 auto-loan
# portfolio. Qwen2.5-7B-Instruct receives six native analytics tools, starts from
# portfolio-level results, and decides for itself which views to request. The model is
# given no information about the planted anomaly: no tool exposes the technical
# month5_default_shock indicator, so any deterioration must be inferred from ordinary
# portfolio measures.
# Requires the previous cell to have been executed (its functions and `outputs`).

import json
import re

from transformers import AutoTokenizer, GenerationConfig, pipeline

# 1. Load the model. Qwen2.5-7B-Instruct (~15 GB in bf16: use GPU-backed compute for
#    this cell; on a CPU-only session it will load but every generation will be very
#    slow) natively renders the `tools` argument in its
#    chat template and replies with its <tool_call {json} </tool_call protocol; this
#    transformers version does not auto-parse it, so step 6 parses it manually.
#    The tokenizer is loaded with clean_up_tokenization_spaces=False (cleanup is
#    destructive for this BPE tokenizer and only produced a warning), and all
#    generation parameters live in a single GenerationConfig object: passing them as
#    pipeline kwargs alongside the model's own generation_config is deprecated, and
#    leaving max_length at its default of 20 collides with max_new_tokens.
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
_qwen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, clean_up_tokenization_spaces=False)
qwen_pipeline = pipeline("text-generation", model=MODEL_NAME, tokenizer=_qwen_tokenizer, device_map="auto")
AGENT_GENERATION_CONFIG = GenerationConfig(max_new_tokens=1024, do_sample=True, temperature=0.1, max_length=None)

# 2. Rebuild the monthly portfolio in memory (deterministic: the fixed SEED reproduces
#    the previous cell's data exactly). Months 1-4 are scored with the same models
#    trained on months 1-2; month 5 is the shocked, scored frame produced above.
_snapshots = generate_snapshots(6000)
_pdm, _lgdm, _features = train(_snapshots)
portfolio = {m: score(_snapshots[m - 1], _pdm, _lgdm, _features) for m in range(1, 5)}
portfolio[5] = outputs["synthetic_auto_loans_month_5_scored"]

GROUP_KEYS = {
    "brand": ["car_brand"],
    "vehicle_condition": ["vehicle_condition"],
    "brand_and_condition": ["car_brand", "vehicle_condition"],
    "stage": ["stage"],
}

# 3. Read-only tool implementations. Python checks the allow-list, validates every
#    argument inside the tool, and pandas performs the calculation; the JSON result goes
#    back through the native tool role. The model can never execute code, submit pandas
#    expressions, read files, modify the frames, or choose arbitrary column names --
#    group keys, brands and conditions are checked against fixed vocabularies, and
#    month5_default_shock is not exposed by any tool.

def _check_month(value, label):
    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 5:
        raise ValueError(f"{label} must be an integer month number from 1 to 5; received {value!r}")
    return value

def _check_group_by(value):
    if value not in GROUP_KEYS:
        raise ValueError(f"group_by must be one of {sorted(GROUP_KEYS)}; received {value!r}")
    return value

def _segment_series(df, group_by):
    keys = GROUP_KEYS[group_by]
    if len(keys) == 2:
        return df[keys[0]].astype(str) + " / " + df[keys[1]].astype(str)
    return df[keys[0]].astype(str)

def _records(df):
    # to_json normalises numpy scalars and NaN -> null, keeping results JSON-serializable
    return json.loads(df.to_json(orient="records"))

def get_available_data():
    return {
        "description": "Monthly snapshots of a synthetic auto-loan portfolio scored under IFRS 9. The same accounts are tracked across all months.",
        "months": [{"month": m, "reporting_date": str(portfolio[m].reporting_date.iloc[0].date()), "accounts": int(len(portfolio[m]))} for m in sorted(portfolio)],
        "permitted_analyses": [
            "Portfolio summary of one month: accounts, default rate, stage distribution, EAD, ECL, modelled PD, modelled LGD and arrears",
            "Comparison of two months by brand, vehicle condition, brand and condition, or stage",
            "Attribution of the ECL movement between two months to those segments",
            "Stage transition matrix between two months, optionally filtered by brand or vehicle condition",
            "New defaults between two months among accounts not already in default at the start",
        ],
    }

def get_portfolio_summary(month):
    _check_month(month, "month")
    d = portfolio[month]
    return {
        "month": month,
        "reporting_date": str(d.reporting_date.iloc[0].date()),
        "accounts": int(len(d)),
        "default_rate": round(float(d.default_status.mean()), 4),
        "stage_accounts": {f"stage_{s}": int((d.stage == s).sum()) for s in (1, 2, 3)},
        "total_ead": round(float(d.ead.sum()), 2),
        "total_ecl": round(float(d.ecl.sum()), 2),
        "avg_modelled_pd": round(float(d.modelled_pd.mean()), 4),
        "avg_modelled_lgd": round(float(d.modelled_lgd.mean()), 4),
        "avg_ifrs9_pd": round(float(d.ifrs9_pd.mean()), 4),
        "arrears": {
            "pct_accounts_in_arrears": round(float((d.days_past_due > 0).mean()), 4),
            "avg_days_past_due": round(float(d.days_past_due.mean()), 2),
            "pct_dpd_30_plus": round(float((d.days_past_due >= 30).mean()), 4),
        },
    }

def _segment_metrics(df, group_by):
    g = df.groupby(_segment_series(df, group_by))
    return pd.DataFrame({
        "accounts": g.size(),
        "default_rate": g.default_status.mean(),
        "total_ead": g.ead.sum(),
        "total_ecl": g.ecl.sum(),
        "avg_modelled_pd": g.modelled_pd.mean(),
        "avg_modelled_lgd": g.modelled_lgd.mean(),
    })

def compare_months(start_month, end_month, group_by):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)
    a = _segment_metrics(portfolio[start_month], group_by).add_suffix("_start")
    b = _segment_metrics(portfolio[end_month], group_by).add_suffix("_end")
    j = a.join(b, how="outer").fillna(0.0)
    j["accounts_start"], j["accounts_end"] = j.accounts_start.astype(int), j.accounts_end.astype(int)
    j["ecl_change"] = j.total_ecl_end - j.total_ecl_start
    j["ecl_change_pct"] = j.ecl_change / j.total_ecl_start.replace(0, pd.NA) * 100
    return {"start_month": start_month, "end_month": end_month, "group_by": group_by,
            "segments": _records(j.rename_axis("segment").reset_index())}

def get_ecl_movement_by_segment(start_month, end_month):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    total_change = float(portfolio[end_month].ecl.sum() - portfolio[start_month].ecl.sum())
    breakdowns = {}
    for gb in ("brand", "vehicle_condition", "brand_and_condition", "stage"):
        a = portfolio[start_month].groupby(_segment_series(portfolio[start_month], gb)).ecl.sum().rename("ecl_start")
        b = portfolio[end_month].groupby(_segment_series(portfolio[end_month], gb)).ecl.sum().rename("ecl_end")
        j = pd.concat([a, b], axis=1).fillna(0.0)
        j["ecl_change"] = j.ecl_end - j.ecl_start
        j["ecl_change_pct"] = j.ecl_change / j.ecl_start.replace(0, pd.NA) * 100
        j["share_of_total_ecl_change_pct"] = j.ecl_change / total_change * 100 if total_change != 0 else pd.NA
        breakdowns[gb] = _records(j.rename_axis("segment").reset_index())
    return {"start_month": start_month, "end_month": end_month,
            "total_ecl_change": round(total_change, 2), "breakdowns": breakdowns}

def get_stage_transition_matrix(start_month, end_month, brand=None, vehicle_condition=None):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    a, b, filters = portfolio[start_month], portfolio[end_month], {}
    if brand is not None:
        if brand not in BRANDS:
            raise ValueError(f"brand must be one of {BRANDS}; received {brand!r}")
        filters["brand"] = brand
        a, b = a[a.car_brand == brand], b[b.car_brand == brand]
    if vehicle_condition is not None:
        if vehicle_condition not in ("New", "Used"):
            raise ValueError(f"vehicle_condition must be 'New' or 'Used'; received {vehicle_condition!r}")
        filters["vehicle_condition"] = vehicle_condition
        a, b = a[a.vehicle_condition == vehicle_condition], b[b.vehicle_condition == vehicle_condition]
    x = a[["account_id", "stage"]].rename(columns={"stage": "from_stage"}).merge(
        b[["account_id", "stage"]].rename(columns={"stage": "to_stage"}), on="account_id")
    ct = pd.crosstab(x.from_stage, x.to_stage)
    matrix = {str(fs): {str(ts): int(n) for ts, n in row.items()} for fs, row in ct.iterrows()}
    return {"start_month": start_month, "end_month": end_month, "filters": filters,
            "accounts": int(len(x)), "transition_matrix": matrix}

def get_new_default_analysis(start_month, end_month, group_by="brand_and_condition"):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)
    a, b = portfolio[start_month], portfolio[end_month]
    x = a.loc[a.default_status == 0, ["account_id"]].merge(b[["account_id", "default_status"]], on="account_id")
    x["segment"] = x.account_id.map(dict(zip(a.account_id, _segment_series(a, group_by))))
    by_segment = x.groupby("segment").agg(accounts=("account_id", "size"),
                                          new_defaults=("default_status", "sum"),
                                          new_default_rate=("default_status", "mean")).reset_index()
    by_segment["new_default_rate"] = by_segment.new_default_rate.round(4)
    return {"start_month": start_month, "end_month": end_month, "group_by": group_by,
            "overall": {"accounts_not_defaulted_at_start": int(len(x)),
                        "new_defaults": int(x.default_status.sum()),
                        "new_default_rate": round(float(x.default_status.mean()), 4)},
            "by_segment": _records(by_segment)}

TOOL_REGISTRY = {
    "get_available_data": get_available_data,
    "get_portfolio_summary": get_portfolio_summary,
    "compare_months": compare_months,
    "get_ecl_movement_by_segment": get_ecl_movement_by_segment,
    "get_stage_transition_matrix": get_stage_transition_matrix,
    "get_new_default_analysis": get_new_default_analysis,
}

# 4. Tool schemas handed to the model. Explicit JSON schemas, so the vocabulary the
#    model sees is exactly the vocabulary the tools validate against.
TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "get_available_data",
        "description": "Describe the available monthly portfolio data: the months, their reporting dates and account counts, and the permitted analyses.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {
        "name": "get_portfolio_summary",
        "description": "Return portfolio-level credit-risk metrics for one month: accounts, default rate, stage distribution, EAD, ECL, modelled PD, modelled LGD and arrears measures.",
        "parameters": {"type": "object", "properties": {
            "month": {"type": "integer", "description": "Month number to summarise (1 to 5)."}},
            "required": ["month"]}}},
    {"type": "function", "function": {
        "name": "compare_months",
        "description": "Compare two months segment by segment. For each segment, returns accounts, default rate, EAD, ECL, modelled PD and modelled LGD in both months, plus the change in ECL.",
        "parameters": {"type": "object", "properties": {
            "start_month": {"type": "integer", "description": "Earlier month number (1 to 5)."},
            "end_month": {"type": "integer", "description": "Later month number (1 to 5)."},
            "group_by": {"type": "string", "enum": ["brand", "vehicle_condition", "brand_and_condition", "stage"],
                          "description": "Segmentation for the comparison."}},
            "required": ["start_month", "end_month", "group_by"]}}},
    {"type": "function", "function": {
        "name": "get_ecl_movement_by_segment",
        "description": "Attribute the ECL movement between two months to segments: by brand, by vehicle condition, by brand and condition, and by stage. Each segment shows ECL in both months, the change, and its share of the total movement.",
        "parameters": {"type": "object", "properties": {
            "start_month": {"type": "integer", "description": "Earlier month number (1 to 5)."},
            "end_month": {"type": "integer", "description": "Later month number (1 to 5)."}},
            "required": ["start_month", "end_month"]}}},
    {"type": "function", "function": {
        "name": "get_stage_transition_matrix",
        "description": "Calculate the IFRS 9 stage transition matrix between two months, with optional filters on brand or vehicle condition.",
        "parameters": {"type": "object", "properties": {
            "start_month": {"type": "integer", "description": "Earlier month number (1 to 5)."},
            "end_month": {"type": "integer", "description": "Later month number (1 to 5)."},
            "brand": {"type": "string", "enum": ["Io", "Europa", "Ganymede", "Callisto", "Amalthea"], "description": "Optional brand filter."},
            "vehicle_condition": {"type": "string", "enum": ["New", "Used"], "description": "Optional vehicle-condition filter."}},
            "required": ["start_month", "end_month"]}}},
    {"type": "function", "function": {
        "name": "get_new_default_analysis",
        "description": "Measure new defaults between two months among accounts that were not already in default at the start of the comparison, overall and by segment.",
        "parameters": {"type": "object", "properties": {
            "start_month": {"type": "integer", "description": "Earlier month number (1 to 5)."},
            "end_month": {"type": "integer", "description": "Later month number (1 to 5)."},
            "group_by": {"type": "string", "enum": ["brand", "vehicle_condition", "brand_and_condition", "stage"],
                          "description": "Optional segmentation; defaults to brand_and_condition."}},
            "required": ["start_month", "end_month"]}}},
]

# 5. Prompts. Deliberately neutral: no mention of Europa, used cars, month 5 or any
#    planted shock. The system prompt only sets investigative discipline.
SYSTEM_PROMPT = (
    "You are a credit-risk analyst reviewing monthly IFRS 9 auto-loan portfolio results. "
    "Begin with broad, portfolio-level checks before drilling down. "
    "Drill down into segments only when the evidence you have already seen supports it. "
    "Do not assume a cause for any movement; keep candidate explanations separate as hypotheses. "
    "Quantify material movements with numbers, in absolute and percentage terms. "
    "Clearly distinguish findings (supported by tool results you retrieved) from hypotheses (plausible but not yet verified). "
    "Stop investigating once you have enough evidence, and then answer with an ordinary response summarising the findings, the supporting figures, and any remaining hypotheses. "
    "Always use the provided tools to obtain data; never guess values. Never state a figure you did not retrieve from a tool result; if a statement needs data you have not retrieved, call a tool first. "
    "Compare at least the earliest and the latest month before concluding whether the results are stable."
)

USER_PROMPT = (
    "Review the available monthly IFRS 9 auto-loan portfolio data. Determine whether the "
    "portfolio and credit-risk results are stable over time. If they are not stable, "
    "investigate the data and explain the most important findings you can support."
)

# 6. The bounded investigation loop. Up to 10 rounds: Qwen requests a tool, Python
#    checks the allow-list and validates the arguments, pandas computes the result, and
#    the JSON goes back through the native tool role. Every requested tool, argument and
#    returned result is printed. An ordinary response (no tool call) ends the loop.
TOOL_CALL_PATTERN = re.compile(r"<tool_call\s*>?\s*(\{.*?\})\s*</tool_call\s*>?", re.DOTALL)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]
print(f"User prompt: {USER_PROMPT}\n")

MAX_ROUNDS = 10
used_tools = set()          # analytical tools the agent has actually retrieved data through
retrieval_pushbacks = 0      # final answers rejected for lacking any retrieved metrics
for round_no in range(1, MAX_ROUNDS + 1):
    response = qwen_pipeline(messages, tools=TOOL_SCHEMAS, generation_config=AGENT_GENERATION_CONFIG)
    assistant_content = response[0]["generated_text"][-1]["content"] or ""
    messages.append({"role": "assistant", "content": assistant_content})
    tool_matches = list(TOOL_CALL_PATTERN.finditer(assistant_content))
    if not tool_matches:
        # Anti-hallucination guard: a small local model sometimes concludes immediately,
        # inventing segment figures it never retrieved. A final answer is accepted
        # only after at least one analytical tool (not just the data inventory) has
        # been used; otherwise the agent is sent back to the tools.
        if used_tools - {"get_available_data"}:
            print("=" * 80)
            print(f"FINAL ANSWER after {round_no} round(s), no further tool requested:")
            print(assistant_content)
            break
        retrieval_pushbacks += 1
        print(f"[round {round_no}] final answer rejected: no analytical data retrieved yet; sending the agent back to the tools.\n")
        messages.append({"role": "user", "content": "You have only described the available data. Retrieve portfolio metrics with the tools (for example, monthly portfolio summaries and month-to-month comparisons) before drawing any conclusion, and never state figures you have not retrieved."})
        continue
    for match in tool_matches:
        name, result = "unknown", None
        try:
            request = json.loads(match.group(1))
            name = request.get("name", "unknown")
            arguments = request.get("arguments", {})
            if name not in TOOL_REGISTRY:
                result = {"error": f"Tool is not permitted: {name!r}. Allowed tools: {sorted(TOOL_REGISTRY)}"}
            else:
                print(f"[round {round_no}] {name} requested with arguments: {json.dumps(arguments)}")
                result = TOOL_REGISTRY[name](**arguments)
                used_tools.add(name)
        except Exception as exc:
            result = {"error": f"{type(exc).__name__}: {exc}"}
        payload = json.dumps(result)
        print(f"[round {round_no}] {name} result: {payload}\n")
        messages.append({"role": "tool", "name": name, "content": payload})
else:
    print("=" * 80)
    print(f"Round limit of {MAX_ROUNDS} reached without a final answer ({retrieval_pushbacks} rejected for missing retrieval); stopping the investigation.")

In [0]:
# Purpose: Run a bounded, read-only, single-prompt IFRS 9 portfolio investigation with Qwen2.5-7B-Instruct and evidence-gated Pandas tools.
# Prerequisite: Run the previous notebook cell first so generate_snapshots, train, score, outputs, BRANDS and pd are available.

import json
import re
import threading

from transformers import AutoTokenizer, GenerationConfig, pipeline


# 1. Load Qwen2.5-7B-Instruct.
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

_qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    clean_up_tokenization_spaces=False,
)

qwen_pipeline = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=_qwen_tokenizer,
    device_map="auto",
)

AGENT_GENERATION_CONFIG = GenerationConfig(
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.1,
    max_length=None,
)

print(f"Loaded model: {MODEL_NAME}")
print(f"Tokenizer context limit: {_qwen_tokenizer.model_max_length:,} tokens")
print(
    "Configured position limit: "
    f"{qwen_pipeline.model.config.max_position_embeddings:,} tokens"
)
print("Model device map:")
print(qwen_pipeline.model.hf_device_map)


# 2. Rebuild the monthly portfolio in memory.
_snapshots = generate_snapshots(6000)
_pdm, _lgdm, _features = train(_snapshots)
portfolio = {
    month: score(_snapshots[month - 1], _pdm, _lgdm, _features)
    for month in range(1, 5)
}
portfolio[5] = outputs["synthetic_auto_loans_month_5_scored"]

GROUP_KEYS = {
    "brand": ["car_brand"],
    "vehicle_condition": ["vehicle_condition"],
    "brand_and_condition": ["car_brand", "vehicle_condition"],
    "stage": ["stage"],
}


# 3. Read-only Pandas tool implementations.
def _check_month(value, label):
    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 5:
        raise ValueError(
            f"{label} must be an integer month number from 1 to 5; received {value!r}"
        )
    return value


def _check_group_by(value):
    if value not in GROUP_KEYS:
        raise ValueError(
            f"group_by must be one of {sorted(GROUP_KEYS)}; received {value!r}"
        )
    return value


def _segment_series(df, group_by):
    keys = GROUP_KEYS[group_by]
    if len(keys) == 2:
        return df[keys[0]].astype(str) + " / " + df[keys[1]].astype(str)
    return df[keys[0]].astype(str)


def _records(df):
    return json.loads(df.to_json(orient="records"))


def get_available_data():
    return {
        "description": (
            "Monthly snapshots of a synthetic auto-loan portfolio scored under IFRS 9. "
            "The same accounts are tracked across all months."
        ),
        "months": [
            {
                "month": month,
                "reporting_date": str(portfolio[month].reporting_date.iloc[0].date()),
                "accounts": int(len(portfolio[month])),
            }
            for month in sorted(portfolio)
        ],
        "permitted_analyses": [
            "Portfolio summary of one month: accounts, default rate, stage distribution, EAD, ECL, modelled PD, modelled LGD and arrears",
            "Comparison of two months by brand, vehicle condition, brand and condition, or stage",
            "Attribution of the ECL movement between two months to those segments",
            "Stage transition matrix between two months, optionally filtered by brand or vehicle condition",
            "New defaults between two months among accounts not already in default at the start",
        ],
    }


def get_portfolio_summary(month):
    _check_month(month, "month")
    data = portfolio[month]
    return {
        "month": month,
        "reporting_date": str(data.reporting_date.iloc[0].date()),
        "accounts": int(len(data)),
        "default_rate": round(float(data.default_status.mean()), 4),
        "stage_accounts": {
            f"stage_{stage}": int((data.stage == stage).sum())
            for stage in (1, 2, 3)
        },
        "total_ead": round(float(data.ead.sum()), 2),
        "total_ecl": round(float(data.ecl.sum()), 2),
        "avg_modelled_pd": round(float(data.modelled_pd.mean()), 4),
        "avg_modelled_lgd": round(float(data.modelled_lgd.mean()), 4),
        "avg_ifrs9_pd": round(float(data.ifrs9_pd.mean()), 4),
        "arrears": {
            "pct_accounts_in_arrears": round(
                float((data.days_past_due > 0).mean()), 4
            ),
            "avg_days_past_due": round(float(data.days_past_due.mean()), 2),
            "pct_dpd_30_plus": round(
                float((data.days_past_due >= 30).mean()), 4
            ),
        },
    }


def _segment_metrics(df, group_by):
    grouped = df.groupby(_segment_series(df, group_by))
    return pd.DataFrame({
        "accounts": grouped.size(),
        "default_rate": grouped.default_status.mean(),
        "total_ead": grouped.ead.sum(),
        "total_ecl": grouped.ecl.sum(),
        "avg_modelled_pd": grouped.modelled_pd.mean(),
        "avg_modelled_lgd": grouped.modelled_lgd.mean(),
    })


def compare_months(start_month, end_month, group_by):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)

    start = _segment_metrics(portfolio[start_month], group_by).add_suffix("_start")
    end = _segment_metrics(portfolio[end_month], group_by).add_suffix("_end")
    joined = start.join(end, how="outer").fillna(0.0)
    joined["accounts_start"] = joined.accounts_start.astype(int)
    joined["accounts_end"] = joined.accounts_end.astype(int)
    joined["ecl_change"] = joined.total_ecl_end - joined.total_ecl_start
    joined["ecl_change_pct"] = (
        joined.ecl_change / joined.total_ecl_start.replace(0, pd.NA) * 100
    )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "group_by": group_by,
        "segments": _records(joined.rename_axis("segment").reset_index()),
    }


def get_ecl_movement_by_segment(start_month, end_month):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")

    total_change = float(
        portfolio[end_month].ecl.sum() - portfolio[start_month].ecl.sum()
    )
    breakdowns = {}

    for group_by in ("brand", "vehicle_condition", "brand_and_condition", "stage"):
        start = (
            portfolio[start_month]
            .groupby(_segment_series(portfolio[start_month], group_by))
            .ecl.sum()
            .rename("ecl_start")
        )
        end = (
            portfolio[end_month]
            .groupby(_segment_series(portfolio[end_month], group_by))
            .ecl.sum()
            .rename("ecl_end")
        )
        joined = pd.concat([start, end], axis=1).fillna(0.0)
        joined["ecl_change"] = joined.ecl_end - joined.ecl_start
        joined["ecl_change_pct"] = (
            joined.ecl_change / joined.ecl_start.replace(0, pd.NA) * 100
        )
        joined["share_of_total_ecl_change_pct"] = (
            joined.ecl_change / total_change * 100
            if total_change != 0
            else pd.NA
        )
        breakdowns[group_by] = _records(
            joined.rename_axis("segment").reset_index()
        )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "total_ecl_change": round(total_change, 2),
        "breakdowns": breakdowns,
    }


def get_stage_transition_matrix(
    start_month,
    end_month,
    brand=None,
    vehicle_condition=None,
):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")

    start = portfolio[start_month]
    end = portfolio[end_month]
    filters = {}

    if brand is not None:
        if brand not in BRANDS:
            raise ValueError(f"brand must be one of {BRANDS}; received {brand!r}")
        filters["brand"] = brand
        start = start[start.car_brand == brand]
        end = end[end.car_brand == brand]

    if vehicle_condition is not None:
        if vehicle_condition not in ("New", "Used"):
            raise ValueError(
                "vehicle_condition must be 'New' or 'Used'; "
                f"received {vehicle_condition!r}"
            )
        filters["vehicle_condition"] = vehicle_condition
        start = start[start.vehicle_condition == vehicle_condition]
        end = end[end.vehicle_condition == vehicle_condition]

    transitions = (
        start[["account_id", "stage"]]
        .rename(columns={"stage": "from_stage"})
        .merge(
            end[["account_id", "stage"]].rename(columns={"stage": "to_stage"}),
            on="account_id",
        )
    )
    table = pd.crosstab(transitions.from_stage, transitions.to_stage)
    matrix = {
        str(from_stage): {str(to_stage): int(count) for to_stage, count in row.items()}
        for from_stage, row in table.iterrows()
    }

    return {
        "start_month": start_month,
        "end_month": end_month,
        "filters": filters,
        "accounts": int(len(transitions)),
        "transition_matrix": matrix,
    }


def get_new_default_analysis(
    start_month,
    end_month,
    group_by="brand_and_condition",
):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)

    start = portfolio[start_month]
    end = portfolio[end_month]
    joined = (
        start.loc[start.default_status == 0, ["account_id"]]
        .merge(end[["account_id", "default_status"]], on="account_id")
    )
    joined["segment"] = joined.account_id.map(
        dict(zip(start.account_id, _segment_series(start, group_by)))
    )
    by_segment = (
        joined.groupby("segment")
        .agg(
            accounts=("account_id", "size"),
            new_defaults=("default_status", "sum"),
            new_default_rate=("default_status", "mean"),
        )
        .reset_index()
    )
    by_segment["new_default_rate"] = by_segment.new_default_rate.round(4)

    return {
        "start_month": start_month,
        "end_month": end_month,
        "group_by": group_by,
        "overall": {
            "accounts_not_defaulted_at_start": int(len(joined)),
            "new_defaults": int(joined.default_status.sum()),
            "new_default_rate": round(float(joined.default_status.mean()), 4),
        },
        "by_segment": _records(by_segment),
    }


TOOL_REGISTRY = {
    "get_available_data": get_available_data,
    "get_portfolio_summary": get_portfolio_summary,
    "compare_months": compare_months,
    "get_ecl_movement_by_segment": get_ecl_movement_by_segment,
    "get_stage_transition_matrix": get_stage_transition_matrix,
    "get_new_default_analysis": get_new_default_analysis,
}


# 4. Explicit JSON tool schemas.
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_available_data",
            "description": "Describe the available months and permitted analyses.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_portfolio_summary",
            "description": "Return portfolio-level risk and ECL metrics for one month.",
            "parameters": {
                "type": "object",
                "properties": {
                    "month": {
                        "type": "integer",
                        "description": "Month number to summarise, from 1 to 5.",
                    }
                },
                "required": ["month"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "compare_months",
            "description": "Compare two months segment by segment.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                    "group_by": {
                        "type": "string",
                        "enum": [
                            "brand",
                            "vehicle_condition",
                            "brand_and_condition",
                            "stage",
                        ],
                    },
                },
                "required": ["start_month", "end_month", "group_by"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_ecl_movement_by_segment",
            "description": "Attribute ECL movement to brands, conditions, combinations and stages.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_stage_transition_matrix",
            "description": "Calculate stage transitions with optional brand and condition filters.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                    "brand": {"type": "string", "enum": BRANDS},
                    "vehicle_condition": {
                        "type": "string",
                        "enum": ["New", "Used"],
                    },
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_new_default_analysis",
            "description": "Analyse new defaults among accounts not defaulted at the start.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                    "group_by": {
                        "type": "string",
                        "enum": [
                            "brand",
                            "vehicle_condition",
                            "brand_and_condition",
                            "stage",
                        ],
                    },
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
]


# 5. Neutral investigation prompts.
SYSTEM_PROMPT = (
    "You are a credit-risk analyst reviewing monthly IFRS 9 auto-loan portfolio results. "
    "Begin with broad, portfolio-level checks before drilling down. "
    "Drill down into segments only when the evidence you have already seen supports it. "
    "Do not assume a cause for any movement; keep candidate explanations separate as hypotheses. "
    "Quantify material movements with numbers, in absolute and percentage terms. "
    "Clearly distinguish findings supported by tool results from hypotheses. "
    "Always use the provided tools to obtain data; never guess values. "
    "Never state a figure you did not retrieve from a tool result. "
    "Compare at least the earliest and latest month before concluding whether results are stable. "
    "A complete investigation must include portfolio-level evidence, segment-level evidence, "
    "and stage-transition or new-default evidence. If more analysis is needed, invoke the native "
    "tool directly rather than describing or recommending the call in prose."
)

USER_PROMPT = (
    "Review the available monthly IFRS 9 auto-loan portfolio data. Determine whether the "
    "portfolio and credit-risk results are stable over time. If they are not stable, "
    "investigate the data and explain the most important findings you can support."
)


# 6. Bounded single-prompt investigation loop with evidence-gated completion.
TOOL_CALL_PATTERN = re.compile(
    r"<tool_call\s*>?\s*(\{.*?\})\s*</tool_call\s*>?",
    re.DOTALL,
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

print(f"User prompt: {USER_PROMPT}\n")

MAX_ROUNDS = 15
used_tools = set()
retrieval_pushbacks = 0


def _heartbeat(stop_event):
    elapsed = 0
    while not stop_event.wait(5):
        elapsed += 5
        print(
            f"[heartbeat] {elapsed}s elapsed - Qwen investigation is still running",
            flush=True,
        )


_stop_event = threading.Event()
_heartbeat_thread = threading.Thread(
    target=_heartbeat,
    args=(_stop_event,),
    daemon=True,
)
_heartbeat_thread.start()

try:
    for round_no in range(1, MAX_ROUNDS + 1):
        # Report current context usage before generation.
        rendered_prompt = _qwen_tokenizer.apply_chat_template(
            messages,
            tools=TOOL_SCHEMAS,
            tokenize=False,
            add_generation_prompt=True,
        )
        input_token_count = len(
            _qwen_tokenizer(
                rendered_prompt,
                add_special_tokens=False,
            )["input_ids"]
        )
        context_limit = qwen_pipeline.model.config.max_position_embeddings
        remaining_context = (
            context_limit
            - input_token_count
            - AGENT_GENERATION_CONFIG.max_new_tokens
        )
        print(
            f"[round {round_no}] context usage: {input_token_count:,} input tokens; "
            f"{remaining_context:,} remain after reserving "
            f"{AGENT_GENERATION_CONFIG.max_new_tokens:,} output tokens."
        )
        if remaining_context < 0:
            raise RuntimeError(
                "The accumulated conversation exceeds the configured context window."
            )

        response = qwen_pipeline(
            messages,
            tools=TOOL_SCHEMAS,
            generation_config=AGENT_GENERATION_CONFIG,
        )
        assistant_content = response[0]["generated_text"][-1]["content"] or ""
        messages.append({"role": "assistant", "content": assistant_content})
        tool_matches = list(TOOL_CALL_PATTERN.finditer(assistant_content))

        has_portfolio_evidence = "get_portfolio_summary" in used_tools
        has_segment_evidence = bool(
            used_tools
            & {
                "compare_months",
                "get_ecl_movement_by_segment",
            }
        )
        has_migration_evidence = bool(
            used_tools
            & {
                "get_stage_transition_matrix",
                "get_new_default_analysis",
            }
        )
        investigation_complete = (
            has_portfolio_evidence
            and has_segment_evidence
            and has_migration_evidence
        )

        if not tool_matches:
            if investigation_complete:
                print("=" * 80)
                print(
                    f"FINAL ANSWER after {round_no} round(s), "
                    "no further tool requested:"
                )
                print(assistant_content)
                break

            missing_evidence = []
            if not has_portfolio_evidence:
                missing_evidence.append("portfolio-level evidence")
            if not has_segment_evidence:
                missing_evidence.append("segment-level evidence")
            if not has_migration_evidence:
                missing_evidence.append(
                    "stage-transition or new-default evidence"
                )

            retrieval_pushbacks += 1
            print(
                f"[round {round_no}] answer or prose-only next step rejected; "
                f"missing: {', '.join(missing_evidence)}.\n"
            )
            messages.append({
                "role": "user",
                "content": (
                    "The investigation is incomplete. Missing evidence: "
                    f"{', '.join(missing_evidence)}. Continue by invoking the "
                    "appropriate native tool now. Do not merely describe or recommend "
                    "the next tool call in prose, and do not provide the final answer yet."
                ),
            })
            continue

        for match in tool_matches:
            name = "unknown"
            result = None
            arguments = {}
            try:
                request = json.loads(match.group(1))
                name = request.get("name", "unknown")
                arguments = request.get("arguments", {})
                if isinstance(arguments, str):
                    arguments = json.loads(arguments)

                if name not in TOOL_REGISTRY:
                    result = {
                        "error": f"Tool is not permitted: {name!r}.",
                        "allowed_tools": sorted(TOOL_REGISTRY),
                    }
                else:
                    print(
                        f"[round {round_no}] {name} requested with arguments: "
                        f"{json.dumps(arguments)}"
                    )
                    result = TOOL_REGISTRY[name](**arguments)
                    used_tools.add(name)
            except Exception as error:
                result = {"error": f"{type(error).__name__}: {error}"}

            payload = json.dumps(result)
            print(f"[round {round_no}] {name} result: {payload}\n")
            messages.append({
                "role": "tool",
                "name": name,
                "content": payload,
            })
    else:
        print("=" * 80)
        print(
            f"Round limit of {MAX_ROUNDS} reached without a final answer "
            f"({retrieval_pushbacks} incomplete responses rejected); "
            "stopping the investigation."
        )
finally:
    _stop_event.set()
    _heartbeat_thread.join(timeout=1)


In [0]:
# Purpose: Install the quantisation backend required to load Qwen2.5-7B in 4-bit.

%pip install -U bitsandbytes accelerate

In [0]:
# Purpose: Run a bounded, read-only, single-prompt IFRS 9 portfolio investigation with Qwen2.5-7B-Instruct and evidence-gated Pandas tools.
# Prerequisite: Run the previous notebook cell first so generate_snapshots, train, score, outputs, BRANDS and pd are available.

import json
import re
import threading

import torch
from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline,
)


# 1. Load Qwen2.5-7B-Instruct.
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

_qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    clean_up_tokenization_spaces=False,
)

_quantisation_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

qwen_pipeline = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=_qwen_tokenizer,
    device_map={"": 0},
    model_kwargs={
        "quantization_config": _quantisation_config,
        "torch_dtype": torch.float16,
    },
)

AGENT_GENERATION_CONFIG = GenerationConfig(
    max_new_tokens=768,
    do_sample=True,
    temperature=0.1,
    max_length=None,
)

print(f"Loaded model: {MODEL_NAME}")
print(f"Tokenizer context limit: {_qwen_tokenizer.model_max_length:,} tokens")
print(
    "Configured position limit: "
    f"{qwen_pipeline.model.config.max_position_embeddings:,} tokens"
)
print("Model device:", next(qwen_pipeline.model.parameters()).device)
print(
    "Loaded in 4-bit:",
    getattr(qwen_pipeline.model, "is_loaded_in_4bit", False),
)
print(
    "Model memory footprint:",
    round(qwen_pipeline.model.get_memory_footprint() / 1024**3, 2),
    "GB",
)


# 2. Rebuild the monthly portfolio in memory.
_snapshots = generate_snapshots(6000)
_pdm, _lgdm, _features = train(_snapshots)
portfolio = {
    month: score(_snapshots[month - 1], _pdm, _lgdm, _features)
    for month in range(1, 5)
}
portfolio[5] = outputs["synthetic_auto_loans_month_5_scored"]

GROUP_KEYS = {
    "brand": ["car_brand"],
    "vehicle_condition": ["vehicle_condition"],
    "brand_and_condition": ["car_brand", "vehicle_condition"],
    "stage": ["stage"],
}


# 3. Read-only Pandas tool implementations.
def _check_month(value, label):
    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 5:
        raise ValueError(
            f"{label} must be an integer month number from 1 to 5; received {value!r}"
        )
    return value


def _check_group_by(value):
    if value not in GROUP_KEYS:
        raise ValueError(
            f"group_by must be one of {sorted(GROUP_KEYS)}; received {value!r}"
        )
    return value


def _segment_series(df, group_by):
    keys = GROUP_KEYS[group_by]
    if len(keys) == 2:
        return df[keys[0]].astype(str) + " / " + df[keys[1]].astype(str)
    return df[keys[0]].astype(str)


def _records(df):
    return json.loads(df.to_json(orient="records"))


def get_available_data():
    return {
        "description": (
            "Monthly snapshots of a synthetic auto-loan portfolio scored under IFRS 9. "
            "The same accounts are tracked across all months."
        ),
        "months": [
            {
                "month": month,
                "reporting_date": str(portfolio[month].reporting_date.iloc[0].date()),
                "accounts": int(len(portfolio[month])),
            }
            for month in sorted(portfolio)
        ],
        "permitted_analyses": [
            "Portfolio summary of one month: accounts, default rate, stage distribution, EAD, ECL, modelled PD, modelled LGD and arrears",
            "Comparison of two months by brand, vehicle condition, brand and condition, or stage",
            "Attribution of the ECL movement between two months to those segments",
            "Stage transition matrix between two months, optionally filtered by brand or vehicle condition",
            "New defaults between two months among accounts not already in default at the start",
        ],
    }


def get_portfolio_summary(month):
    _check_month(month, "month")
    data = portfolio[month]
    return {
        "month": month,
        "reporting_date": str(data.reporting_date.iloc[0].date()),
        "accounts": int(len(data)),
        "default_rate": round(float(data.default_status.mean()), 4),
        "stage_accounts": {
            f"stage_{stage}": int((data.stage == stage).sum())
            for stage in (1, 2, 3)
        },
        "total_ead": round(float(data.ead.sum()), 2),
        "total_ecl": round(float(data.ecl.sum()), 2),
        "avg_modelled_pd": round(float(data.modelled_pd.mean()), 4),
        "avg_modelled_lgd": round(float(data.modelled_lgd.mean()), 4),
        "avg_ifrs9_pd": round(float(data.ifrs9_pd.mean()), 4),
        "arrears": {
            "pct_accounts_in_arrears": round(
                float((data.days_past_due > 0).mean()), 4
            ),
            "avg_days_past_due": round(float(data.days_past_due.mean()), 2),
            "pct_dpd_30_plus": round(
                float((data.days_past_due >= 30).mean()), 4
            ),
        },
    }


def _segment_metrics(df, group_by):
    grouped = df.groupby(_segment_series(df, group_by))
    return pd.DataFrame({
        "accounts": grouped.size(),
        "default_rate": grouped.default_status.mean(),
        "total_ead": grouped.ead.sum(),
        "total_ecl": grouped.ecl.sum(),
        "avg_modelled_pd": grouped.modelled_pd.mean(),
        "avg_modelled_lgd": grouped.modelled_lgd.mean(),
    })


def compare_months(start_month, end_month, group_by):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)

    start = _segment_metrics(portfolio[start_month], group_by).add_suffix("_start")
    end = _segment_metrics(portfolio[end_month], group_by).add_suffix("_end")
    joined = start.join(end, how="outer").fillna(0.0)
    joined["accounts_start"] = joined.accounts_start.astype(int)
    joined["accounts_end"] = joined.accounts_end.astype(int)
    joined["ecl_change"] = joined.total_ecl_end - joined.total_ecl_start
    joined["ecl_change_pct"] = (
        joined.ecl_change / joined.total_ecl_start.replace(0, pd.NA) * 100
    )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "group_by": group_by,
        "segments": _records(joined.rename_axis("segment").reset_index()),
    }


def get_ecl_movement_by_segment(start_month, end_month):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")

    total_change = float(
        portfolio[end_month].ecl.sum() - portfolio[start_month].ecl.sum()
    )
    breakdowns = {}

    for group_by in ("brand", "vehicle_condition", "brand_and_condition", "stage"):
        start = (
            portfolio[start_month]
            .groupby(_segment_series(portfolio[start_month], group_by))
            .ecl.sum()
            .rename("ecl_start")
        )
        end = (
            portfolio[end_month]
            .groupby(_segment_series(portfolio[end_month], group_by))
            .ecl.sum()
            .rename("ecl_end")
        )
        joined = pd.concat([start, end], axis=1).fillna(0.0)
        joined["ecl_change"] = joined.ecl_end - joined.ecl_start
        joined["ecl_change_pct"] = (
            joined.ecl_change / joined.ecl_start.replace(0, pd.NA) * 100
        )
        joined["share_of_total_ecl_change_pct"] = (
            joined.ecl_change / total_change * 100
            if total_change != 0
            else pd.NA
        )
        breakdowns[group_by] = _records(
            joined.rename_axis("segment").reset_index()
        )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "total_ecl_change": round(total_change, 2),
        "breakdowns": breakdowns,
    }


def get_stage_transition_matrix(
    start_month,
    end_month,
    brand=None,
    vehicle_condition=None,
):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")

    start = portfolio[start_month]
    end = portfolio[end_month]
    filters = {}

    if brand is not None:
        if brand not in BRANDS:
            raise ValueError(f"brand must be one of {BRANDS}; received {brand!r}")
        filters["brand"] = brand
        start = start[start.car_brand == brand]
        end = end[end.car_brand == brand]

    if vehicle_condition is not None:
        if vehicle_condition not in ("New", "Used"):
            raise ValueError(
                "vehicle_condition must be 'New' or 'Used'; "
                f"received {vehicle_condition!r}"
            )
        filters["vehicle_condition"] = vehicle_condition
        start = start[start.vehicle_condition == vehicle_condition]
        end = end[end.vehicle_condition == vehicle_condition]

    transitions = (
        start[["account_id", "stage"]]
        .rename(columns={"stage": "from_stage"})
        .merge(
            end[["account_id", "stage"]].rename(columns={"stage": "to_stage"}),
            on="account_id",
        )
    )
    table = pd.crosstab(transitions.from_stage, transitions.to_stage)
    matrix = {
        str(from_stage): {str(to_stage): int(count) for to_stage, count in row.items()}
        for from_stage, row in table.iterrows()
    }

    return {
        "start_month": start_month,
        "end_month": end_month,
        "filters": filters,
        "accounts": int(len(transitions)),
        "transition_matrix": matrix,
    }


def get_new_default_analysis(
    start_month,
    end_month,
    group_by="brand_and_condition",
):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)

    start = portfolio[start_month]
    end = portfolio[end_month]
    joined = (
        start.loc[start.default_status == 0, ["account_id"]]
        .merge(end[["account_id", "default_status"]], on="account_id")
    )
    joined["segment"] = joined.account_id.map(
        dict(zip(start.account_id, _segment_series(start, group_by)))
    )
    by_segment = (
        joined.groupby("segment")
        .agg(
            accounts=("account_id", "size"),
            new_defaults=("default_status", "sum"),
            new_default_rate=("default_status", "mean"),
        )
        .reset_index()
    )
    by_segment["new_default_rate"] = by_segment.new_default_rate.round(4)

    return {
        "start_month": start_month,
        "end_month": end_month,
        "group_by": group_by,
        "overall": {
            "accounts_not_defaulted_at_start": int(len(joined)),
            "new_defaults": int(joined.default_status.sum()),
            "new_default_rate": round(float(joined.default_status.mean()), 4),
        },
        "by_segment": _records(by_segment),
    }


TOOL_REGISTRY = {
    "get_available_data": get_available_data,
    "get_portfolio_summary": get_portfolio_summary,
    "compare_months": compare_months,
    "get_ecl_movement_by_segment": get_ecl_movement_by_segment,
    "get_stage_transition_matrix": get_stage_transition_matrix,
    "get_new_default_analysis": get_new_default_analysis,
}


# 4. Explicit JSON tool schemas.
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_available_data",
            "description": "Describe the available months and permitted analyses.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_portfolio_summary",
            "description": "Return portfolio-level risk and ECL metrics for one month.",
            "parameters": {
                "type": "object",
                "properties": {
                    "month": {
                        "type": "integer",
                        "description": "Month number to summarise, from 1 to 5.",
                    }
                },
                "required": ["month"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "compare_months",
            "description": "Compare two months segment by segment.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                    "group_by": {
                        "type": "string",
                        "enum": [
                            "brand",
                            "vehicle_condition",
                            "brand_and_condition",
                            "stage",
                        ],
                    },
                },
                "required": ["start_month", "end_month", "group_by"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_ecl_movement_by_segment",
            "description": "Attribute ECL movement to brands, conditions, combinations and stages.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_stage_transition_matrix",
            "description": "Calculate stage transitions with optional brand and condition filters.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                    "brand": {"type": "string", "enum": BRANDS},
                    "vehicle_condition": {
                        "type": "string",
                        "enum": ["New", "Used"],
                    },
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_new_default_analysis",
            "description": "Analyse new defaults among accounts not defaulted at the start.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                    "group_by": {
                        "type": "string",
                        "enum": [
                            "brand",
                            "vehicle_condition",
                            "brand_and_condition",
                            "stage",
                        ],
                    },
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
]


# 5. Neutral investigation prompts.
SYSTEM_PROMPT = (
    "You are a credit-risk analyst reviewing monthly IFRS 9 auto-loan portfolio results. "
    "Begin with broad, portfolio-level checks before drilling down. "
    "Drill down into segments only when the evidence you have already seen supports it. "
    "Do not assume a cause for any movement; keep candidate explanations separate as hypotheses. "
    "Quantify material movements with numbers, in absolute and percentage terms. "
    "Clearly distinguish findings supported by tool results from hypotheses. "
    "Always use the provided tools to obtain data; never guess values. "
    "Never state a figure you did not retrieve from a tool result. "
    "Compare at least the earliest and latest month before concluding whether results are stable. "
    "A complete investigation must include portfolio-level evidence, segment-level evidence, "
    "and stage-transition or new-default evidence. If more analysis is needed, invoke the native "
    "tool directly rather than describing or recommending the call in prose. "
    "The only valid group_by values are brand, vehicle_condition, "
    "brand_and_condition and stage; there is no portfolio group_by value."
)

USER_PROMPT = (
    "Review the available monthly IFRS 9 auto-loan portfolio data. Determine whether the "
    "portfolio and credit-risk results are stable over time. If they are not stable, "
    "investigate the data and explain the most important findings you can support."
)


# 6. Bounded single-prompt investigation loop with evidence-gated completion.
TOOL_CALL_PATTERN = re.compile(
    r"<tool_call\s*>?\s*(\{.*?\})\s*</tool_call\s*>?",
    re.DOTALL,
)


def _extract_balanced_json(text, start_index):
    """Extract one complete JSON object beginning at or after start_index."""
    object_start = text.find("{", start_index)
    if object_start < 0:
        return None

    depth = 0
    in_string = False
    escaped = False

    for index in range(object_start, len(text)):
        character = text[index]

        if in_string:
            if escaped:
                escaped = False
            elif character == "\\":
                escaped = True
            elif character == '"':
                in_string = False
            continue

        if character == '"':
            in_string = True
        elif character == "{":
            depth += 1
        elif character == "}":
            depth -= 1
            if depth == 0:
                return text[object_start:index + 1]

    return None


def _parse_tool_requests(assistant_content):
    """Parse closed or unclosed Qwen tool calls without treating them as final prose."""
    requests = []

    for match in TOOL_CALL_PATTERN.finditer(assistant_content):
        requests.append(json.loads(match.group(1)))

    if requests:
        return requests

    marker_pattern = re.compile(r"<tool_call\s*>?", re.IGNORECASE)
    for marker in marker_pattern.finditer(assistant_content):
        json_payload = _extract_balanced_json(assistant_content, marker.end())
        if json_payload is not None:
            requests.append(json.loads(json_payload))

    return requests

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

print(f"User prompt: {USER_PROMPT}\n")

MAX_ROUNDS = 15
used_tools = set()
retrieval_pushbacks = 0


def _heartbeat(stop_event):
    elapsed = 0
    while not stop_event.wait(5):
        elapsed += 5
        print(
            f"[heartbeat] {elapsed}s elapsed - Qwen investigation is still running",
            flush=True,
        )


_stop_event = threading.Event()
_heartbeat_thread = threading.Thread(
    target=_heartbeat,
    args=(_stop_event,),
    daemon=True,
)
_heartbeat_thread.start()

try:
    for round_no in range(1, MAX_ROUNDS + 1):
        # Report current context usage before generation.
        rendered_prompt = _qwen_tokenizer.apply_chat_template(
            messages,
            tools=TOOL_SCHEMAS,
            tokenize=False,
            add_generation_prompt=True,
        )
        input_token_count = len(
            _qwen_tokenizer(
                rendered_prompt,
                add_special_tokens=False,
            )["input_ids"]
        )
        context_limit = qwen_pipeline.model.config.max_position_embeddings
        remaining_context = (
            context_limit
            - input_token_count
            - AGENT_GENERATION_CONFIG.max_new_tokens
        )
        print(
            f"[round {round_no}] context usage: {input_token_count:,} input tokens; "
            f"{remaining_context:,} remain after reserving "
            f"{AGENT_GENERATION_CONFIG.max_new_tokens:,} output tokens."
        )
        if remaining_context < 0:
            raise RuntimeError(
                "The accumulated conversation exceeds the configured context window."
            )

        response = qwen_pipeline(
            messages,
            tools=TOOL_SCHEMAS,
            generation_config=AGENT_GENERATION_CONFIG,
        )
        assistant_content = response[0]["generated_text"][-1]["content"] or ""
        messages.append({"role": "assistant", "content": assistant_content})
        tool_requests = _parse_tool_requests(assistant_content)
        contains_tool_marker = "<tool_call" in assistant_content.lower()

        has_portfolio_evidence = "get_portfolio_summary" in used_tools
        has_segment_evidence = bool(
            used_tools
            & {
                "compare_months",
                "get_ecl_movement_by_segment",
            }
        )
        has_migration_evidence = bool(
            used_tools
            & {
                "get_stage_transition_matrix",
                "get_new_default_analysis",
            }
        )
        investigation_complete = (
            has_portfolio_evidence
            and has_segment_evidence
            and has_migration_evidence
        )

        if not tool_requests:
            if contains_tool_marker:
                retrieval_pushbacks += 1
                print(
                    f"[round {round_no}] incomplete or malformed native tool call "
                    "detected; asking Qwen to reissue it.\n"
                )
                messages.append({
                    "role": "user",
                    "content": (
                        "Your previous response contained an incomplete or malformed "
                        "native tool call. Reissue exactly one valid native tool call now. "
                        "Use only an allowed group_by value: brand, vehicle_condition, "
                        "brand_and_condition, or stage. Do not add prose."
                    ),
                })
                continue

            if investigation_complete:
                print("=" * 80)
                print(
                    f"FINAL ANSWER after {round_no} round(s), "
                    "no further tool requested:"
                )
                print(assistant_content)
                break

            missing_evidence = []
            if not has_portfolio_evidence:
                missing_evidence.append("portfolio-level evidence")
            if not has_segment_evidence:
                missing_evidence.append("segment-level evidence")
            if not has_migration_evidence:
                missing_evidence.append(
                    "stage-transition or new-default evidence"
                )

            retrieval_pushbacks += 1
            print(
                f"[round {round_no}] answer or prose-only next step rejected; "
                f"missing: {', '.join(missing_evidence)}.\n"
            )
            messages.append({
                "role": "user",
                "content": (
                    "The investigation is incomplete. Missing evidence: "
                    f"{', '.join(missing_evidence)}. Continue by invoking the "
                    "appropriate native tool now. Do not merely describe or recommend "
                    "the next tool call in prose, and do not provide the final answer yet."
                ),
            })
            continue

        for request in tool_requests:
            name = "unknown"
            result = None
            arguments = {}
            try:
                name = request.get("name", "unknown")
                arguments = request.get("arguments", {})
                if isinstance(arguments, str):
                    arguments = json.loads(arguments)

                if name not in TOOL_REGISTRY:
                    result = {
                        "error": f"Tool is not permitted: {name!r}.",
                        "allowed_tools": sorted(TOOL_REGISTRY),
                    }
                else:
                    print(
                        f"[round {round_no}] {name} requested with arguments: "
                        f"{json.dumps(arguments)}"
                    )
                    result = TOOL_REGISTRY[name](**arguments)
                    used_tools.add(name)
            except Exception as error:
                result = {"error": f"{type(error).__name__}: {error}"}

            payload = json.dumps(result)
            print(f"[round {round_no}] {name} result: {payload}\n")
            messages.append({
                "role": "tool",
                "name": name,
                "content": payload,
            })
    else:
        print("=" * 80)
        print(
            f"Round limit of {MAX_ROUNDS} reached without a final answer "
            f"({retrieval_pushbacks} incomplete responses rejected); "
            "stopping the investigation."
        )
finally:
    _stop_event.set()
    _heartbeat_thread.join(timeout=1)


In [0]:
# Purpose: Display cumulative default rates by month, brand and vehicle condition with the segment labels visible in Databricks.

import pandas as pd


portfolio_all_months = pd.concat(
    [
        portfolio[month].assign(month=month)
        for month in sorted(portfolio)
    ],
    ignore_index=True
)


default_rate_long = (
    portfolio_all_months
    .groupby(
        [
            "month",
            "car_brand",
            "vehicle_condition"
        ],
        observed=True
    )
    .agg(
        accounts=("account_id", "nunique"),
        defaults=("default_status", "sum")
    )
    .reset_index()
)


default_rate_long["default_rate_pct"] = (
    default_rate_long["defaults"]
    / default_rate_long["accounts"]
    * 100
).round(2)


default_rate_long["brand_and_condition"] = (
    default_rate_long["car_brand"]
    + " / "
    + default_rate_long["vehicle_condition"]
)


default_rate_crosstab = (
    default_rate_long
    .pivot(
        index="brand_and_condition",
        columns="month",
        values="default_rate_pct"
    )
    .rename(
        columns=lambda month: f"Month {month} default rate (%)"
    )
    .reset_index()
    .rename(
        columns={
            "brand_and_condition": "Brand / Vehicle condition"
        }
    )
    .sort_values(
        "Brand / Vehicle condition"
    )
    .reset_index(drop=True)
)


print(
    "Cumulative default rate by month, brand "
    "and vehicle condition:\n"
)

display(default_rate_crosstab)

In [0]:
# Purpose: Generate five in-memory synthetic auto-loan snapshots with stable segment default rates in months 1-4, shock only Europa used cars in month 5, train PD/LGD models on months 1-2, and calculate account-level ECL.

import threading

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 20261120
BRANDS = ["Io", "Europa", "Ganymede", "Callisto", "Amalthea"]
DATES = pd.to_datetime([
    "2026-01-31",
    "2026-02-28",
    "2026-03-31",
    "2026-04-30",
    "2026-05-31",
])
TARGET_BRAND = "Europa"
TARGET_CONDITION = "Used"


def heartbeat(stop_event):
    """Print one heartbeat and a concise progress update every five seconds."""
    elapsed = 0
    while not stop_event.wait(5):
        elapsed += 5
        print(
            f"[heartbeat] still working - {elapsed}s elapsed; "
            "stable portfolio generation and month-5 shock are progressing",
            flush=True,
        )


def sigmoid(value):
    return 1.0 / (1.0 + np.exp(-np.clip(value, -30, 30)))


def generate_base_accounts(n_accounts=6000, seed=SEED):
    """Create one deliberately unrealistic synthetic auto-loan population."""
    rng = np.random.default_rng(seed)
    condition = rng.choice(["New", "Used"], n_accounts, p=[0.35, 0.65])
    vehicle_price = np.where(
        condition == "New",
        rng.normal(4500, 450, n_accounts),
        rng.normal(300, 55, n_accounts),
    ).clip(100)
    term_months = rng.choice(
        [12, 24, 36, 48, 60],
        n_accounts,
        p=[0.08, 0.17, 0.30, 0.27, 0.18],
    )
    start_dates = pd.Timestamp("2021-01-01") + pd.to_timedelta(
        rng.integers(0, 1809, n_accounts),
        unit="D",
    )
    deposit_pct = rng.uniform(0.0, 0.35, n_accounts)

    base = pd.DataFrame({
        "account_id": [f"MOON-{i:06d}" for i in range(1, n_accounts + 1)],
        "car_brand": rng.choice(BRANDS, n_accounts),
        "vehicle_condition": condition,
        "vehicle_price": vehicle_price.round(2),
        "loan_start_date": start_dates,
        "original_term_months": term_months,
        "original_balance": (
            vehicle_price * (1.0 - deposit_pct)
            + rng.uniform(25, 250, n_accounts)
        ).round(2),
        "interest_rate": rng.uniform(0.03, 0.42, n_accounts).round(5),
        "borrower_age": rng.integers(18, 86, n_accounts),
        "annual_income": rng.lognormal(
            np.log(18000), 0.65, n_accounts
        ).clip(3000, 120000).round(2),
        "credit_score": rng.normal(560, 95, n_accounts)
        .clip(250, 850)
        .round()
        .astype(int),
        "deposit_pct": deposit_pct.round(5),
    })

    # Establish one baseline default population. These exact account-level default
    # statuses are held constant in months 1-4, which keeps every brand-condition
    # segment's cumulative default rate stable until the deliberate month-5 shock.
    baseline_default_probability = sigmoid(
        -4.15
        + 0.010 * (520 - base["credit_score"])
        + 1.00 * base["vehicle_condition"].eq("Used").astype(float)
        + 1.25 * (base["interest_rate"] - 0.12)
        + 0.000012 * (18000 - base["annual_income"])
    )
    base["baseline_default_status"] = rng.binomial(
        1,
        baseline_default_probability,
    ).astype(int)

    return base


def generate_snapshots(n=6000, seed=SEED):
    """Generate five monthly DataFrames with stable defaults in months 1-4."""
    rng = np.random.default_rng(seed + 1)
    base = generate_base_accounts(n, seed)
    snapshots = []

    # Create one persistent non-default arrears tendency so month-to-month movement
    # remains modest without creating new defaults before the shock.
    arrears_propensity = sigmoid(
        -3.1
        + 0.009 * (600 - base["credit_score"])
        + 1.8 * (base["interest_rate"] - 0.12)
        + 0.000012 * (22000 - base["annual_income"])
    )
    persistent_arrears = rng.binomial(1, arrears_propensity)
    persistent_dpd_band = rng.choice(
        [5, 15, 35, 65],
        n,
        p=[0.25, 0.30, 0.30, 0.15],
    )

    for month_no, reporting_date in enumerate(DATES, start=1):
        frame = base.drop(columns=["baseline_default_status"]).copy(deep=True)
        frame["month_no"] = month_no
        frame["reporting_date"] = reporting_date
        frame["months_on_book"] = (
            (reporting_date.year - frame["loan_start_date"].dt.year) * 12
            + reporting_date.month
            - frame["loan_start_date"].dt.month
        ).clip(lower=0)
        frame["remaining_term_months"] = (
            frame["original_term_months"] - frame["months_on_book"]
        ).clip(lower=1)

        runoff = (
            frame["months_on_book"] / frame["original_term_months"]
        ).clip(0, 0.92)
        frame["outstanding_balance"] = (
            frame["original_balance"]
            * (1.0 - runoff)
            * rng.normal(1.0, 0.012, n)
        ).clip(20).round(2)

        month_noise = rng.integers(-2, 3, n)
        frame["days_past_due"] = np.where(
            persistent_arrears == 1,
            persistent_dpd_band + month_noise,
            0,
        ).clip(0)

        # The same accounts remain defaulted throughout months 1-4 and initially
        # in month 5. No segment can develop an increasing default rate by accident.
        frame["default_status"] = base["baseline_default_status"].to_numpy()
        frame.loc[
            frame["default_status"].eq(1),
            "days_past_due",
        ] = np.maximum(
            frame.loc[frame["default_status"].eq(1), "days_past_due"],
            90,
        )

        frame["stage"] = np.select(
            [
                frame["default_status"].eq(1),
                frame["days_past_due"].ge(30),
            ],
            [3, 2],
            default=1,
        ).astype(int)
        frame["ead"] = (
            frame["outstanding_balance"]
            * (1.0 + 0.08 * frame["stage"].ge(2))
        ).round(2)
        frame["collateral_value"] = (
            frame["vehicle_price"]
            * np.where(frame["vehicle_condition"].eq("New"), 0.72, 0.38)
            * np.exp(-0.015 * frame["months_on_book"])
        ).clip(10).round(2)
        frame["loan_to_value"] = (
            frame["ead"] / frame["collateral_value"]
        ).clip(0, 20).round(5)

        recovery_rate = sigmoid(
            1.0
            - 0.32 * frame["loan_to_value"]
            - 0.012 * frame["days_past_due"]
            + 0.35 * frame["vehicle_condition"].eq("New").astype(float)
            + rng.normal(0, 0.25, n)
        )
        frame["observed_recovery_amount"] = np.where(
            frame["default_status"].eq(1),
            frame["ead"] * recovery_rate,
            np.nan,
        ).round(2)
        frame["observed_lgd"] = np.where(
            frame["default_status"].eq(1),
            1.0 - frame["observed_recovery_amount"] / frame["ead"],
            np.nan,
        )
        frame["month5_default_shock"] = 0
        snapshots.append(frame)

    return snapshots


def prep(categorical, numerical):
    return ColumnTransformer([
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical,
        ),
        (
            "numerical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numerical,
        ),
    ])


def train(snapshots):
    """Train PD and LGD models on months 1 and 2 only."""
    training = pd.concat(snapshots[:2], ignore_index=True)
    categorical = ["car_brand", "vehicle_condition"]
    numerical = [
        "vehicle_price",
        "original_term_months",
        "interest_rate",
        "borrower_age",
        "annual_income",
        "credit_score",
        "deposit_pct",
        "months_on_book",
        "remaining_term_months",
        "outstanding_balance",
        "days_past_due",
        "collateral_value",
        "loan_to_value",
        "ead",
    ]
    features = categorical + numerical

    pd_model = Pipeline([
        ("preprocessor", prep(categorical, numerical)),
        (
            "model",
            LogisticRegression(
                max_iter=1500,
                class_weight="balanced",
                random_state=SEED,
            ),
        ),
    ]).fit(training[features], training["default_status"])

    default_training = training[training["default_status"].eq(1)]
    if len(default_training) < 50:
        raise RuntimeError(
            f"Insufficient defaults to train the LGD model: {len(default_training)}"
        )

    lgd_model = Pipeline([
        ("preprocessor", prep(categorical, numerical)),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=150,
                learning_rate=0.06,
                max_depth=4,
                random_state=SEED,
            ),
        ),
    ]).fit(default_training[features], default_training["observed_lgd"])

    return pd_model, lgd_model, features


def apply_month5_shock(month4, month5, brand=TARGET_BRAND):
    """Double the cumulative default count only for used cars of one brand."""
    rng = np.random.default_rng(SEED + 5)
    shocked = month5.copy(deep=True)

    target = (
        shocked["car_brand"].eq(brand)
        & shocked["vehicle_condition"].eq(TARGET_CONDITION)
    )
    baseline_defaults = target & shocked["default_status"].eq(1)
    baseline_default_count = int(baseline_defaults.sum())
    candidates = shocked.index[
        target & shocked["default_status"].eq(0)
    ].to_numpy()

    additional_default_count = min(
        baseline_default_count,
        len(candidates),
    )
    selected = rng.choice(
        candidates,
        size=additional_default_count,
        replace=False,
    )

    shocked.loc[selected, "month5_default_shock"] = 1
    shocked.loc[selected, "default_status"] = 1
    shocked.loc[selected, "days_past_due"] = np.maximum(
        shocked.loc[selected, "days_past_due"],
        90,
    )
    shocked.loc[selected, "stage"] = 3
    shocked.loc[selected, "ead"] = (
        shocked.loc[selected, "outstanding_balance"] * 1.08
    ).round(2)

    # Give the newly defaulted accounts observed recovery/LGD values for completeness.
    recovery_rate = sigmoid(
        1.0
        - 0.32 * shocked.loc[selected, "loan_to_value"]
        - 0.012 * shocked.loc[selected, "days_past_due"]
        + rng.normal(0, 0.25, additional_default_count)
    )
    shocked.loc[selected, "observed_recovery_amount"] = (
        shocked.loc[selected, "ead"] * recovery_rate
    ).round(2)
    shocked.loc[selected, "observed_lgd"] = (
        1.0
        - shocked.loc[selected, "observed_recovery_amount"]
        / shocked.loc[selected, "ead"]
    )

    target_accounts = int(target.sum())
    diagnostics = {
        "brand": brand,
        "vehicle_condition": TARGET_CONDITION,
        "target_accounts": target_accounts,
        "baseline_defaults": baseline_default_count,
        "additional_defaults": additional_default_count,
        "month5_defaults_after_shock": baseline_default_count
        + additional_default_count,
        "baseline_default_rate_pct": round(
            baseline_default_count / target_accounts * 100,
            2,
        ),
        "month5_default_rate_pct": round(
            (baseline_default_count + additional_default_count)
            / target_accounts
            * 100,
            2,
        ),
    }
    return shocked, diagnostics


def score(frame, pd_model, lgd_model, features):
    """Score one monthly snapshot and calculate account-level ECL."""
    scored = frame.copy(deep=True)
    scored["modelled_pd"] = pd_model.predict_proba(scored[features])[:, 1].clip(
        0.0001,
        0.9999,
    )
    scored["modelled_lgd"] = lgd_model.predict(scored[features]).clip(0.05, 0.95)
    multiplier = np.where(
        scored["stage"].eq(1),
        1.0,
        np.minimum(scored["remaining_term_months"] / 12.0, 5.0),
    )
    scored["ifrs9_pd"] = (scored["modelled_pd"] * multiplier).clip(0.0001, 1.0)
    scored.loc[scored["stage"].eq(3), "ifrs9_pd"] = 1.0
    scored["ecl"] = (
        scored["ifrs9_pd"] * scored["modelled_lgd"] * scored["ead"]
    ).round(2)
    return scored


def build_outputs(n_accounts=6000):
    """Build all five scored DataFrames and validation tables entirely in memory."""
    snapshots = generate_snapshots(n_accounts)
    snapshots[4], shock_diagnostics = apply_month5_shock(
        snapshots[3],
        snapshots[4],
    )
    pd_model, lgd_model, features = train(snapshots)
    scored = {
        month: score(snapshot, pd_model, lgd_model, features)
        for month, snapshot in enumerate(snapshots, start=1)
    }

    all_months = pd.concat(
        [frame.assign(month=month) for month, frame in scored.items()],
        ignore_index=True,
    )
    default_rate_detail = (
        all_months
        .groupby(
            ["month", "car_brand", "vehicle_condition"],
            observed=True,
        )
        .agg(
            accounts=("account_id", "nunique"),
            defaults=("default_status", "sum"),
        )
        .reset_index()
    )
    default_rate_detail["default_rate_pct"] = (
        default_rate_detail["defaults"]
        / default_rate_detail["accounts"]
        * 100
    ).round(2)
    default_rate_detail["segment"] = (
        default_rate_detail["car_brand"]
        + " / "
        + default_rate_detail["vehicle_condition"]
    )
    default_rate_crosstab = (
        default_rate_detail
        .pivot(index="segment", columns="month", values="default_rate_pct")
        .rename(columns=lambda month: f"Month {month} default rate (%)")
        .reset_index()
        .rename(columns={"segment": "Brand / Vehicle condition"})
        .sort_values("Brand / Vehicle condition")
        .reset_index(drop=True)
    )

    # Hard validation: months 1-4 must be exactly stable for every segment.
    stable_columns = [
        "Month 1 default rate (%)",
        "Month 2 default rate (%)",
        "Month 3 default rate (%)",
        "Month 4 default rate (%)",
    ]
    stable_range = default_rate_crosstab[stable_columns].max(axis=1) - default_rate_crosstab[stable_columns].min(axis=1)
    if not np.allclose(stable_range.to_numpy(), 0.0):
        raise AssertionError("At least one segment's default rate changed before month 5.")

    target_row = default_rate_crosstab[
        default_rate_crosstab["Brand / Vehicle condition"].eq("Europa / Used")
    ].iloc[0]
    non_target = default_rate_crosstab[
        ~default_rate_crosstab["Brand / Vehicle condition"].eq("Europa / Used")
    ]
    if not np.allclose(
        non_target["Month 5 default rate (%)"].to_numpy(),
        non_target["Month 4 default rate (%)"].to_numpy(),
    ):
        raise AssertionError("A non-target segment changed default rate in month 5.")
    if target_row["Month 5 default rate (%)"] <= target_row["Month 4 default rate (%)"]:
        raise AssertionError("Europa / Used did not show the intended month-5 shock.")

    outputs = {
        "synthetic_auto_loans_month_1_scored": scored[1],
        "synthetic_auto_loans_month_2_scored": scored[2],
        "synthetic_auto_loans_month_3_scored": scored[3],
        "synthetic_auto_loans_month_4_scored": scored[4],
        "synthetic_auto_loans_month_5_scored": scored[5],
        "scored_portfolio": scored,
        "pd_model": pd_model,
        "lgd_model": lgd_model,
        "model_features": features,
        "shock_diagnostics": shock_diagnostics,
        "default_rate_detail": default_rate_detail,
        "default_rate_crosstab": default_rate_crosstab,
    }
    return outputs


stop_event = threading.Event()
heartbeat_thread = threading.Thread(
    target=heartbeat,
    args=(stop_event,),
    daemon=True,
)
heartbeat_thread.start()

try:
    outputs = build_outputs(6000)
    portfolio = outputs["scored_portfolio"]

    print("\nShock diagnostics:")
    print(pd.Series(outputs["shock_diagnostics"]).to_string())

    print("\nDefault-rate validation table:")
    print(outputs["default_rate_crosstab"].to_string(index=False))

    # In Databricks, display the labelled table interactively when display is available.
    if "display" in globals():
        display(outputs["default_rate_crosstab"])
finally:
    stop_event.set()
    heartbeat_thread.join(timeout=1)


In [0]:
# Purpose: Run a bounded, read-only, single-prompt IFRS 9 portfolio investigation with Qwen2.5-7B-Instruct and evidence-gated Pandas tools.
# Prerequisite: Run the previous notebook cell first so generate_snapshots, train, score, outputs, BRANDS and pd are available.

import json
import re
import threading

import torch
from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline,
)


# 1. Load Qwen2.5-7B-Instruct.
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

_qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    clean_up_tokenization_spaces=False,
)

_quantisation_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

qwen_pipeline = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=_qwen_tokenizer,
    device_map={"": 0},
    model_kwargs={
        "quantization_config": _quantisation_config,
        "torch_dtype": torch.float16,
    },
)

AGENT_GENERATION_CONFIG = GenerationConfig(
    max_new_tokens=768,
    do_sample=True,
    temperature=0.1,
    max_length=None,
)

print(f"Loaded model: {MODEL_NAME}")
print(f"Tokenizer context limit: {_qwen_tokenizer.model_max_length:,} tokens")
print(
    "Configured position limit: "
    f"{qwen_pipeline.model.config.max_position_embeddings:,} tokens"
)
print("Model device:", next(qwen_pipeline.model.parameters()).device)
print(
    "Loaded in 4-bit:",
    getattr(qwen_pipeline.model, "is_loaded_in_4bit", False),
)
print(
    "Model memory footprint:",
    round(qwen_pipeline.model.get_memory_footprint() / 1024**3, 2),
    "GB",
)


# 2. Rebuild the monthly portfolio in memory.
_snapshots = generate_snapshots(6000)
_pdm, _lgdm, _features = train(_snapshots)
portfolio = {
    month: score(_snapshots[month - 1], _pdm, _lgdm, _features)
    for month in range(1, 5)
}
portfolio[5] = outputs["synthetic_auto_loans_month_5_scored"]

GROUP_KEYS = {
    "brand": ["car_brand"],
    "vehicle_condition": ["vehicle_condition"],
    "brand_and_condition": ["car_brand", "vehicle_condition"],
    "stage": ["stage"],
}


# 3. Read-only Pandas tool implementations.
def _check_month(value, label):
    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 5:
        raise ValueError(
            f"{label} must be an integer month number from 1 to 5; received {value!r}"
        )
    return value


def _check_group_by(value):
    if value not in GROUP_KEYS:
        raise ValueError(
            f"group_by must be one of {sorted(GROUP_KEYS)}; received {value!r}"
        )
    return value


def _segment_series(df, group_by):
    keys = GROUP_KEYS[group_by]
    if len(keys) == 2:
        return df[keys[0]].astype(str) + " / " + df[keys[1]].astype(str)
    return df[keys[0]].astype(str)


def _records(df):
    return json.loads(df.to_json(orient="records"))


def get_available_data():
    return {
        "description": (
            "Monthly snapshots of a synthetic auto-loan portfolio scored under IFRS 9. "
            "The same accounts are tracked across all months."
        ),
        "months": [
            {
                "month": month,
                "reporting_date": str(portfolio[month].reporting_date.iloc[0].date()),
                "accounts": int(len(portfolio[month])),
            }
            for month in sorted(portfolio)
        ],
        "permitted_analyses": [
            "Portfolio summary of one month: accounts, default rate, stage distribution, EAD, ECL, modelled PD, modelled LGD and arrears",
            "Comparison of two months by brand, vehicle condition, brand and condition, or stage",
            "Attribution of the ECL movement between two months to those segments",
            "Stage transition matrix between two months, optionally filtered by brand or vehicle condition",
            "New defaults between two months among accounts not already in default at the start",
        ],
    }


def get_portfolio_summary(month):
    _check_month(month, "month")
    data = portfolio[month]
    return {
        "month": month,
        "reporting_date": str(data.reporting_date.iloc[0].date()),
        "accounts": int(len(data)),
        "default_rate": round(float(data.default_status.mean()), 4),
        "stage_accounts": {
            f"stage_{stage}": int((data.stage == stage).sum())
            for stage in (1, 2, 3)
        },
        "total_ead": round(float(data.ead.sum()), 2),
        "total_ecl": round(float(data.ecl.sum()), 2),
        "avg_modelled_pd": round(float(data.modelled_pd.mean()), 4),
        "avg_modelled_lgd": round(float(data.modelled_lgd.mean()), 4),
        "avg_ifrs9_pd": round(float(data.ifrs9_pd.mean()), 4),
        "arrears": {
            "pct_accounts_in_arrears": round(
                float((data.days_past_due > 0).mean()), 4
            ),
            "avg_days_past_due": round(float(data.days_past_due.mean()), 2),
            "pct_dpd_30_plus": round(
                float((data.days_past_due >= 30).mean()), 4
            ),
        },
    }


def _segment_metrics(df, group_by):
    grouped = df.groupby(_segment_series(df, group_by))
    return pd.DataFrame({
        "accounts": grouped.size(),
        "default_rate": grouped.default_status.mean(),
        "total_ead": grouped.ead.sum(),
        "total_ecl": grouped.ecl.sum(),
        "avg_modelled_pd": grouped.modelled_pd.mean(),
        "avg_modelled_lgd": grouped.modelled_lgd.mean(),
    })


def compare_months(start_month, end_month, group_by):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)

    start = _segment_metrics(portfolio[start_month], group_by).add_suffix("_start")
    end = _segment_metrics(portfolio[end_month], group_by).add_suffix("_end")
    joined = start.join(end, how="outer").fillna(0.0)
    joined["accounts_start"] = joined.accounts_start.astype(int)
    joined["accounts_end"] = joined.accounts_end.astype(int)
    joined["ecl_change"] = joined.total_ecl_end - joined.total_ecl_start
    joined["ecl_change_pct"] = (
        joined.ecl_change / joined.total_ecl_start.replace(0, pd.NA) * 100
    )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "group_by": group_by,
        "segments": _records(joined.rename_axis("segment").reset_index()),
    }


def get_ecl_movement_by_segment(start_month, end_month):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")

    total_change = float(
        portfolio[end_month].ecl.sum() - portfolio[start_month].ecl.sum()
    )
    breakdowns = {}

    for group_by in ("brand", "vehicle_condition", "brand_and_condition", "stage"):
        start = (
            portfolio[start_month]
            .groupby(_segment_series(portfolio[start_month], group_by))
            .ecl.sum()
            .rename("ecl_start")
        )
        end = (
            portfolio[end_month]
            .groupby(_segment_series(portfolio[end_month], group_by))
            .ecl.sum()
            .rename("ecl_end")
        )
        joined = pd.concat([start, end], axis=1).fillna(0.0)
        joined["ecl_change"] = joined.ecl_end - joined.ecl_start
        joined["ecl_change_pct"] = (
            joined.ecl_change / joined.ecl_start.replace(0, pd.NA) * 100
        )
        joined["share_of_total_ecl_change_pct"] = (
            joined.ecl_change / total_change * 100
            if total_change != 0
            else pd.NA
        )
        breakdowns[group_by] = _records(
            joined.rename_axis("segment").reset_index()
        )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "total_ecl_change": round(total_change, 2),
        "breakdowns": breakdowns,
    }


def get_stage_transition_matrix(
    start_month,
    end_month,
    brand=None,
    vehicle_condition=None,
):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")

    start = portfolio[start_month]
    end = portfolio[end_month]
    filters = {}

    if brand is not None:
        if brand not in BRANDS:
            raise ValueError(f"brand must be one of {BRANDS}; received {brand!r}")
        filters["brand"] = brand
        start = start[start.car_brand == brand]
        end = end[end.car_brand == brand]

    if vehicle_condition is not None:
        if vehicle_condition not in ("New", "Used"):
            raise ValueError(
                "vehicle_condition must be 'New' or 'Used'; "
                f"received {vehicle_condition!r}"
            )
        filters["vehicle_condition"] = vehicle_condition
        start = start[start.vehicle_condition == vehicle_condition]
        end = end[end.vehicle_condition == vehicle_condition]

    transitions = (
        start[["account_id", "stage"]]
        .rename(columns={"stage": "from_stage"})
        .merge(
            end[["account_id", "stage"]].rename(columns={"stage": "to_stage"}),
            on="account_id",
        )
    )
    table = pd.crosstab(transitions.from_stage, transitions.to_stage)
    matrix = {
        str(from_stage): {str(to_stage): int(count) for to_stage, count in row.items()}
        for from_stage, row in table.iterrows()
    }

    return {
        "start_month": start_month,
        "end_month": end_month,
        "filters": filters,
        "accounts": int(len(transitions)),
        "transition_matrix": matrix,
    }


def get_new_default_analysis(
    start_month,
    end_month,
    group_by="brand_and_condition",
):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)

    start = portfolio[start_month]
    end = portfolio[end_month]
    joined = (
        start.loc[start.default_status == 0, ["account_id"]]
        .merge(end[["account_id", "default_status"]], on="account_id")
    )
    joined["segment"] = joined.account_id.map(
        dict(zip(start.account_id, _segment_series(start, group_by)))
    )
    by_segment = (
        joined.groupby("segment")
        .agg(
            accounts=("account_id", "size"),
            new_defaults=("default_status", "sum"),
            new_default_rate=("default_status", "mean"),
        )
        .reset_index()
    )
    by_segment["new_default_rate"] = by_segment.new_default_rate.round(4)

    return {
        "start_month": start_month,
        "end_month": end_month,
        "group_by": group_by,
        "overall": {
            "accounts_not_defaulted_at_start": int(len(joined)),
            "new_defaults": int(joined.default_status.sum()),
            "new_default_rate": round(float(joined.default_status.mean()), 4),
        },
        "by_segment": _records(by_segment),
    }


TOOL_REGISTRY = {
    "get_available_data": get_available_data,
    "get_portfolio_summary": get_portfolio_summary,
    "compare_months": compare_months,
    "get_ecl_movement_by_segment": get_ecl_movement_by_segment,
    "get_stage_transition_matrix": get_stage_transition_matrix,
    "get_new_default_analysis": get_new_default_analysis,
}


# 4. Explicit JSON tool schemas.
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_available_data",
            "description": "Describe the available months and permitted analyses.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_portfolio_summary",
            "description": "Return portfolio-level risk and ECL metrics for one month.",
            "parameters": {
                "type": "object",
                "properties": {
                    "month": {
                        "type": "integer",
                        "description": "Month number to summarise, from 1 to 5.",
                    }
                },
                "required": ["month"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "compare_months",
            "description": "Compare two months segment by segment.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                    "group_by": {
                        "type": "string",
                        "enum": [
                            "brand",
                            "vehicle_condition",
                            "brand_and_condition",
                            "stage",
                        ],
                    },
                },
                "required": ["start_month", "end_month", "group_by"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_ecl_movement_by_segment",
            "description": "Attribute ECL movement to brands, conditions, combinations and stages.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_stage_transition_matrix",
            "description": "Calculate stage transitions with optional brand and condition filters.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                    "brand": {"type": "string", "enum": BRANDS},
                    "vehicle_condition": {
                        "type": "string",
                        "enum": ["New", "Used"],
                    },
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_new_default_analysis",
            "description": "Analyse new defaults among accounts not defaulted at the start.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer"},
                    "end_month": {"type": "integer"},
                    "group_by": {
                        "type": "string",
                        "enum": [
                            "brand",
                            "vehicle_condition",
                            "brand_and_condition",
                            "stage",
                        ],
                    },
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
]


# 5. Neutral investigation prompts.
SYSTEM_PROMPT = (
    "You are a credit-risk analyst reviewing monthly IFRS 9 auto-loan portfolio results. "
    "Begin with broad, portfolio-level checks before drilling down. "
    "Drill down into segments only when the evidence you have already seen supports it. "
    "Do not assume a cause for any movement; keep candidate explanations separate as hypotheses. "
    "Quantify material movements with numbers, in absolute and percentage terms. "
    "Clearly distinguish findings supported by tool results from hypotheses. "
    "Always use the provided tools to obtain data; never guess values. "
    "Never state a figure you did not retrieve from a tool result. "
    "Compare at least the earliest and latest month before concluding whether results are stable. "
    "A complete investigation must include portfolio-level evidence, segment-level evidence, "
    "and stage-transition or new-default evidence. If more analysis is needed, invoke the native "
    "tool directly rather than describing or recommending the call in prose. "
    "The only valid group_by values are brand, vehicle_condition, "
    "brand_and_condition and stage; there is no portfolio group_by value."
)

USER_PROMPT = (
    "Review the available monthly IFRS 9 auto-loan portfolio data. Determine whether the "
    "portfolio and credit-risk results are stable over time. If they are not stable, "
    "investigate the data and explain the most important findings you can support."
)


# 6. Bounded single-prompt investigation loop with evidence-gated completion.
TOOL_CALL_PATTERN = re.compile(
    r"<tool_call\s*>?\s*(\{.*?\})\s*</tool_call\s*>?",
    re.DOTALL,
)


def _extract_balanced_json(text, start_index):
    """Extract one complete JSON object beginning at or after start_index."""
    object_start = text.find("{", start_index)
    if object_start < 0:
        return None

    depth = 0
    in_string = False
    escaped = False

    for index in range(object_start, len(text)):
        character = text[index]

        if in_string:
            if escaped:
                escaped = False
            elif character == "\\":
                escaped = True
            elif character == '"':
                in_string = False
            continue

        if character == '"':
            in_string = True
        elif character == "{":
            depth += 1
        elif character == "}":
            depth -= 1
            if depth == 0:
                return text[object_start:index + 1]

    return None


def _parse_tool_requests(assistant_content):
    """Parse closed or unclosed Qwen tool calls without treating them as final prose."""
    requests = []

    for match in TOOL_CALL_PATTERN.finditer(assistant_content):
        requests.append(json.loads(match.group(1)))

    if requests:
        return requests

    marker_pattern = re.compile(r"<tool_call\s*>?", re.IGNORECASE)
    for marker in marker_pattern.finditer(assistant_content):
        json_payload = _extract_balanced_json(assistant_content, marker.end())
        if json_payload is not None:
            requests.append(json.loads(json_payload))

    return requests

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

print(f"User prompt: {USER_PROMPT}\n")

MAX_ROUNDS = 15
used_tools = set()
retrieval_pushbacks = 0


def _heartbeat(stop_event):
    elapsed = 0
    while not stop_event.wait(5):
        elapsed += 5
        print(
            f"[heartbeat] {elapsed}s elapsed - Qwen investigation is still running",
            flush=True,
        )


_stop_event = threading.Event()
_heartbeat_thread = threading.Thread(
    target=_heartbeat,
    args=(_stop_event,),
    daemon=True,
)
_heartbeat_thread.start()

try:
    for round_no in range(1, MAX_ROUNDS + 1):
        # Report current context usage before generation.
        rendered_prompt = _qwen_tokenizer.apply_chat_template(
            messages,
            tools=TOOL_SCHEMAS,
            tokenize=False,
            add_generation_prompt=True,
        )
        input_token_count = len(
            _qwen_tokenizer(
                rendered_prompt,
                add_special_tokens=False,
            )["input_ids"]
        )
        context_limit = qwen_pipeline.model.config.max_position_embeddings
        remaining_context = (
            context_limit
            - input_token_count
            - AGENT_GENERATION_CONFIG.max_new_tokens
        )
        print(
            f"[round {round_no}] context usage: {input_token_count:,} input tokens; "
            f"{remaining_context:,} remain after reserving "
            f"{AGENT_GENERATION_CONFIG.max_new_tokens:,} output tokens."
        )
        if remaining_context < 0:
            raise RuntimeError(
                "The accumulated conversation exceeds the configured context window."
            )

        response = qwen_pipeline(
            messages,
            tools=TOOL_SCHEMAS,
            generation_config=AGENT_GENERATION_CONFIG,
        )
        assistant_content = response[0]["generated_text"][-1]["content"] or ""
        messages.append({"role": "assistant", "content": assistant_content})
        tool_requests = _parse_tool_requests(assistant_content)
        contains_tool_marker = "<tool_call" in assistant_content.lower()

        has_portfolio_evidence = "get_portfolio_summary" in used_tools
        has_segment_evidence = bool(
            used_tools
            & {
                "compare_months",
                "get_ecl_movement_by_segment",
            }
        )
        has_migration_evidence = bool(
            used_tools
            & {
                "get_stage_transition_matrix",
                "get_new_default_analysis",
            }
        )
        investigation_complete = (
            has_portfolio_evidence
            and has_segment_evidence
            and has_migration_evidence
        )

        if not tool_requests:
            if contains_tool_marker:
                retrieval_pushbacks += 1
                print(
                    f"[round {round_no}] incomplete or malformed native tool call "
                    "detected; asking Qwen to reissue it.\n"
                )
                messages.append({
                    "role": "user",
                    "content": (
                        "Your previous response contained an incomplete or malformed "
                        "native tool call. Reissue exactly one valid native tool call now. "
                        "Use only an allowed group_by value: brand, vehicle_condition, "
                        "brand_and_condition, or stage. Do not add prose."
                    ),
                })
                continue

            if investigation_complete:
                print("=" * 80)
                print(
                    f"FINAL ANSWER after {round_no} round(s), "
                    "no further tool requested:"
                )
                print(assistant_content)
                break

            missing_evidence = []
            if not has_portfolio_evidence:
                missing_evidence.append("portfolio-level evidence")
            if not has_segment_evidence:
                missing_evidence.append("segment-level evidence")
            if not has_migration_evidence:
                missing_evidence.append(
                    "stage-transition or new-default evidence"
                )

            retrieval_pushbacks += 1
            print(
                f"[round {round_no}] answer or prose-only next step rejected; "
                f"missing: {', '.join(missing_evidence)}.\n"
            )
            messages.append({
                "role": "user",
                "content": (
                    "The investigation is incomplete. Missing evidence: "
                    f"{', '.join(missing_evidence)}. Continue by invoking the "
                    "appropriate native tool now. Do not merely describe or recommend "
                    "the next tool call in prose, and do not provide the final answer yet."
                ),
            })
            continue

        for request in tool_requests:
            name = "unknown"
            result = None
            arguments = {}
            try:
                name = request.get("name", "unknown")
                arguments = request.get("arguments", {})
                if isinstance(arguments, str):
                    arguments = json.loads(arguments)

                if name not in TOOL_REGISTRY:
                    result = {
                        "error": f"Tool is not permitted: {name!r}.",
                        "allowed_tools": sorted(TOOL_REGISTRY),
                    }
                else:
                    print(
                        f"[round {round_no}] {name} requested with arguments: "
                        f"{json.dumps(arguments)}"
                    )
                    result = TOOL_REGISTRY[name](**arguments)
                    used_tools.add(name)
            except Exception as error:
                result = {"error": f"{type(error).__name__}: {error}"}

            payload = json.dumps(result)
            print(f"[round {round_no}] {name} result: {payload}\n")
            messages.append({
                "role": "tool",
                "name": name,
                "content": payload,
            })
    else:
        print("=" * 80)
        print(
            f"Round limit of {MAX_ROUNDS} reached without a final answer "
            f"({retrieval_pushbacks} incomplete responses rejected); "
            "stopping the investigation."
        )
finally:
    _stop_event.set()
    _heartbeat_thread.join(timeout=1)


In [0]:
# Purpose: Run a four-phase, evidence-gated IFRS 9 portfolio investigation with Qwen2.5-7B-Instruct in 4-bit and read-only Pandas tools.
# Prerequisite: Run the previous data-generation cell first so generate_snapshots, train, score, outputs, BRANDS and pd are available.

import json
import re
import threading
from copy import deepcopy

import torch
from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline,
)


# -----------------------------------------------------------------------------
# 1. Model loading and deterministic generation
# -----------------------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

_qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    clean_up_tokenization_spaces=False,
)

_quantisation_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

qwen_pipeline = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=_qwen_tokenizer,
    device_map={"": 0},
    model_kwargs={
        "quantization_config": _quantisation_config,
        "torch_dtype": torch.float16,
    },
)

AGENT_GENERATION_CONFIG = GenerationConfig(
    max_new_tokens=768,
    do_sample=False,
    max_length=None,
)

print(f"Loaded model: {MODEL_NAME}")
print(f"Tokenizer context limit: {_qwen_tokenizer.model_max_length:,} tokens")
print(
    "Configured position limit: "
    f"{qwen_pipeline.model.config.max_position_embeddings:,} tokens"
)
print("Model device:", next(qwen_pipeline.model.parameters()).device)
print(
    "Loaded in 4-bit:",
    getattr(qwen_pipeline.model, "is_loaded_in_4bit", False),
)
print(
    "Model memory footprint:",
    round(qwen_pipeline.model.get_memory_footprint() / 1024**3, 2),
    "GB",
)


# -----------------------------------------------------------------------------
# 2. Rebuild the deterministic in-memory portfolio
# -----------------------------------------------------------------------------

_snapshots = generate_snapshots(6000)
_pdm, _lgdm, _features = train(_snapshots)

portfolio = {
    month: score(_snapshots[month - 1], _pdm, _lgdm, _features)
    for month in range(1, 5)
}
portfolio[5] = outputs["synthetic_auto_loans_month_5_scored"]

GROUP_KEYS = {
    "brand": ["car_brand"],
    "vehicle_condition": ["vehicle_condition"],
    "brand_and_condition": ["car_brand", "vehicle_condition"],
    "stage": ["stage"],
}


# -----------------------------------------------------------------------------
# 3. Validation and read-only Pandas tools
# -----------------------------------------------------------------------------

def _check_month(value, label):
    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 5:
        raise ValueError(
            f"{label} must be an integer month number from 1 to 5; received {value!r}"
        )
    return value


def _check_group_by(value):
    if value not in GROUP_KEYS:
        raise ValueError(
            f"group_by must be one of {sorted(GROUP_KEYS)}; received {value!r}"
        )
    return value


def _segment_series(df, group_by):
    keys = GROUP_KEYS[group_by]
    if len(keys) == 2:
        return df[keys[0]].astype(str) + " / " + df[keys[1]].astype(str)
    return df[keys[0]].astype(str)


def _records(df):
    return json.loads(df.to_json(orient="records"))


def _percentage_change(start_value, end_value):
    if start_value == 0:
        return None
    return round((end_value / start_value - 1.0) * 100.0, 2)


def get_available_data():
    return {
        "description": (
            "Monthly snapshots of a synthetic auto-loan portfolio scored under "
            "IFRS 9. The same accounts are tracked across all months."
        ),
        "months": [
            {
                "month": month,
                "reporting_date": str(portfolio[month].reporting_date.iloc[0].date()),
                "accounts": int(len(portfolio[month])),
            }
            for month in sorted(portfolio)
        ],
        "permitted_analyses": [
            "Portfolio summary and movement between any two months",
            "Comparison by brand, vehicle condition, brand and condition, or stage",
            "ECL movement attribution by segment",
            "Stage transition matrix with optional brand and condition filters",
            "New-default analysis among accounts not already defaulted at the start",
        ],
    }


def get_portfolio_summary(month):
    _check_month(month, "month")
    d = portfolio[month]
    default_rate = float(d.default_status.mean())
    avg_pd = float(d.modelled_pd.mean())
    avg_lgd = float(d.modelled_lgd.mean())
    avg_ifrs9_pd = float(d.ifrs9_pd.mean())
    return {
        "month": month,
        "reporting_date": str(d.reporting_date.iloc[0].date()),
        "accounts": int(len(d)),
        "defaults": int(d.default_status.sum()),
        "default_rate_decimal": round(default_rate, 4),
        "default_rate_pct": round(default_rate * 100, 2),
        "stage_accounts": {
            f"stage_{stage}": int((d.stage == stage).sum()) for stage in (1, 2, 3)
        },
        "total_ead": round(float(d.ead.sum()), 2),
        "total_ead_millions": round(float(d.ead.sum()) / 1_000_000, 4),
        "total_ecl": round(float(d.ecl.sum()), 2),
        "total_ecl_thousands": round(float(d.ecl.sum()) / 1_000, 2),
        "avg_modelled_pd_decimal": round(avg_pd, 4),
        "avg_modelled_pd_pct": round(avg_pd * 100, 2),
        "avg_modelled_lgd_decimal": round(avg_lgd, 4),
        "avg_modelled_lgd_pct": round(avg_lgd * 100, 2),
        "avg_ifrs9_pd_decimal": round(avg_ifrs9_pd, 4),
        "avg_ifrs9_pd_pct": round(avg_ifrs9_pd * 100, 2),
        "arrears": {
            "pct_accounts_in_arrears": round(float((d.days_past_due > 0).mean()) * 100, 2),
            "avg_days_past_due": round(float(d.days_past_due.mean()), 2),
            "pct_dpd_30_plus": round(float((d.days_past_due >= 30).mean()) * 100, 2),
        },
    }


def compare_portfolio_totals(start_month, end_month):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    start = portfolio[start_month]
    end = portfolio[end_month]

    start_ead = float(start.ead.sum())
    end_ead = float(end.ead.sum())
    start_ecl = float(start.ecl.sum())
    end_ecl = float(end.ecl.sum())
    start_default_rate = float(start.default_status.mean())
    end_default_rate = float(end.default_status.mean())

    return {
        "start_month": start_month,
        "end_month": end_month,
        "ead": {
            "start": round(start_ead, 2),
            "end": round(end_ead, 2),
            "absolute_change": round(end_ead - start_ead, 2),
            "percentage_change": _percentage_change(start_ead, end_ead),
        },
        "ecl": {
            "start": round(start_ecl, 2),
            "end": round(end_ecl, 2),
            "absolute_change": round(end_ecl - start_ecl, 2),
            "percentage_change": _percentage_change(start_ecl, end_ecl),
        },
        "defaults": {
            "start_count": int(start.default_status.sum()),
            "end_count": int(end.default_status.sum()),
            "absolute_change": int(end.default_status.sum() - start.default_status.sum()),
            "start_rate_pct": round(start_default_rate * 100, 2),
            "end_rate_pct": round(end_default_rate * 100, 2),
            "rate_change_percentage_points": round(
                (end_default_rate - start_default_rate) * 100, 2
            ),
        },
        "stage_3": {
            "start_count": int(start.stage.eq(3).sum()),
            "end_count": int(end.stage.eq(3).sum()),
            "absolute_change": int(end.stage.eq(3).sum() - start.stage.eq(3).sum()),
        },
    }


def _segment_metrics(df, group_by):
    grouped = df.groupby(_segment_series(df, group_by))
    result = pd.DataFrame({
        "accounts": grouped.size(),
        "defaults": grouped.default_status.sum(),
        "default_rate_pct": grouped.default_status.mean() * 100,
        "total_ead": grouped.ead.sum(),
        "total_ecl": grouped.ecl.sum(),
        "avg_modelled_pd_pct": grouped.modelled_pd.mean() * 100,
        "avg_modelled_lgd_pct": grouped.modelled_lgd.mean() * 100,
    })
    return result


def compare_months(start_month, end_month, group_by):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)

    start = _segment_metrics(portfolio[start_month], group_by).add_suffix("_start")
    end = _segment_metrics(portfolio[end_month], group_by).add_suffix("_end")
    joined = start.join(end, how="outer").fillna(0.0)

    joined["accounts_start"] = joined.accounts_start.astype(int)
    joined["accounts_end"] = joined.accounts_end.astype(int)
    joined["defaults_start"] = joined.defaults_start.astype(int)
    joined["defaults_end"] = joined.defaults_end.astype(int)
    joined["default_rate_change_pp"] = (
        joined.default_rate_pct_end - joined.default_rate_pct_start
    )
    joined["ecl_change"] = joined.total_ecl_end - joined.total_ecl_start
    joined["ecl_change_pct"] = (
        joined.ecl_change / joined.total_ecl_start.replace(0, pd.NA) * 100
    )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "group_by": group_by,
        "segments": _records(joined.rename_axis("segment").reset_index()),
    }


def get_ecl_movement_by_segment(start_month, end_month):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")

    total_change = float(
        portfolio[end_month].ecl.sum() - portfolio[start_month].ecl.sum()
    )
    breakdowns = {}

    for group_by in ("brand", "vehicle_condition", "brand_and_condition", "stage"):
        start = (
            portfolio[start_month]
            .groupby(_segment_series(portfolio[start_month], group_by))
            .ecl.sum()
            .rename("ecl_start")
        )
        end = (
            portfolio[end_month]
            .groupby(_segment_series(portfolio[end_month], group_by))
            .ecl.sum()
            .rename("ecl_end")
        )
        joined = pd.concat([start, end], axis=1).fillna(0.0)
        joined["ecl_change"] = joined.ecl_end - joined.ecl_start
        joined["ecl_change_pct"] = (
            joined.ecl_change / joined.ecl_start.replace(0, pd.NA) * 100
        )
        joined["share_of_total_ecl_change_pct"] = (
            joined.ecl_change / total_change * 100 if total_change != 0 else pd.NA
        )
        joined = joined.sort_values("ecl_change", ascending=False)
        breakdowns[group_by] = _records(
            joined.rename_axis("segment").reset_index()
        )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "total_ecl_change": round(total_change, 2),
        "breakdowns": breakdowns,
    }


def get_stage_transition_matrix(
    start_month,
    end_month,
    brand=None,
    vehicle_condition=None,
):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")

    start = portfolio[start_month]
    end = portfolio[end_month]
    filters = {}

    if brand is not None:
        if brand not in BRANDS:
            raise ValueError(f"brand must be one of {BRANDS}; received {brand!r}")
        filters["brand"] = brand
        start = start[start.car_brand == brand]
        end = end[end.car_brand == brand]

    if vehicle_condition is not None:
        if vehicle_condition not in ("New", "Used"):
            raise ValueError(
                "vehicle_condition must be 'New' or 'Used'; "
                f"received {vehicle_condition!r}"
            )
        filters["vehicle_condition"] = vehicle_condition
        start = start[start.vehicle_condition == vehicle_condition]
        end = end[end.vehicle_condition == vehicle_condition]

    transitions = (
        start[["account_id", "stage"]]
        .rename(columns={"stage": "from_stage"})
        .merge(
            end[["account_id", "stage"]].rename(columns={"stage": "to_stage"}),
            on="account_id",
        )
    )
    counts = (
        transitions.groupby(["from_stage", "to_stage"])
        .size()
        .rename("accounts")
        .reset_index()
    )
    counts["from_stage_total"] = counts.groupby("from_stage").accounts.transform("sum")
    counts["row_percentage"] = counts.accounts / counts.from_stage_total * 100

    return {
        "start_month": start_month,
        "end_month": end_month,
        "filters": filters,
        "accounts": int(len(transitions)),
        "transitions": _records(counts),
    }


def get_new_default_analysis(
    start_month,
    end_month,
    group_by="brand_and_condition",
):
    _check_month(start_month, "start_month")
    _check_month(end_month, "end_month")
    _check_group_by(group_by)

    start = portfolio[start_month]
    end = portfolio[end_month]
    joined = (
        start.loc[start.default_status == 0, ["account_id"]]
        .merge(end[["account_id", "default_status", "ecl"]], on="account_id")
    )
    joined["segment"] = joined.account_id.map(
        dict(zip(start.account_id, _segment_series(start, group_by)))
    )

    by_segment = (
        joined.groupby("segment")
        .agg(
            eligible_accounts=("account_id", "size"),
            new_defaults=("default_status", "sum"),
            end_ecl=("ecl", "sum"),
        )
        .reset_index()
    )
    by_segment["new_default_rate_pct"] = (
        by_segment.new_defaults / by_segment.eligible_accounts * 100
    )
    by_segment = by_segment.sort_values(
        ["new_default_rate_pct", "new_defaults"],
        ascending=False,
    )

    return {
        "start_month": start_month,
        "end_month": end_month,
        "group_by": group_by,
        "overall": {
            "accounts_not_defaulted_at_start": int(len(joined)),
            "new_defaults": int(joined.default_status.sum()),
            "new_default_rate_pct": round(float(joined.default_status.mean()) * 100, 2),
        },
        "by_segment": _records(by_segment),
    }


TOOL_REGISTRY = {
    "get_available_data": get_available_data,
    "get_portfolio_summary": get_portfolio_summary,
    "compare_portfolio_totals": compare_portfolio_totals,
    "compare_months": compare_months,
    "get_ecl_movement_by_segment": get_ecl_movement_by_segment,
    "get_stage_transition_matrix": get_stage_transition_matrix,
    "get_new_default_analysis": get_new_default_analysis,
}


# -----------------------------------------------------------------------------
# 4. Explicit tool schemas and phase-specific tool access
# -----------------------------------------------------------------------------

ALL_TOOL_SCHEMAS = {
    "get_available_data": {
        "type": "function",
        "function": {
            "name": "get_available_data",
            "description": "Describe available monthly portfolio data and analyses.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    "get_portfolio_summary": {
        "type": "function",
        "function": {
            "name": "get_portfolio_summary",
            "description": "Return portfolio-level risk and ECL metrics for one month.",
            "parameters": {
                "type": "object",
                "properties": {
                    "month": {"type": "integer", "minimum": 1, "maximum": 5}
                },
                "required": ["month"],
            },
        },
    },
    "compare_portfolio_totals": {
        "type": "function",
        "function": {
            "name": "compare_portfolio_totals",
            "description": "Calculate exact portfolio-level movements between two months.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "end_month": {"type": "integer", "minimum": 1, "maximum": 5},
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    "compare_months": {
        "type": "function",
        "function": {
            "name": "compare_months",
            "description": "Compare two months by an approved segment.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "end_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "group_by": {
                        "type": "string",
                        "enum": ["brand", "vehicle_condition", "brand_and_condition", "stage"],
                    },
                },
                "required": ["start_month", "end_month", "group_by"],
            },
        },
    },
    "get_ecl_movement_by_segment": {
        "type": "function",
        "function": {
            "name": "get_ecl_movement_by_segment",
            "description": "Attribute ECL movement to brands, conditions, combinations and stages.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "end_month": {"type": "integer", "minimum": 1, "maximum": 5},
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    "get_stage_transition_matrix": {
        "type": "function",
        "function": {
            "name": "get_stage_transition_matrix",
            "description": "Calculate stage transitions with optional brand and condition filters.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "end_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "brand": {"type": "string", "enum": BRANDS},
                    "vehicle_condition": {"type": "string", "enum": ["New", "Used"]},
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
    "get_new_default_analysis": {
        "type": "function",
        "function": {
            "name": "get_new_default_analysis",
            "description": "Analyse new defaults among accounts not defaulted at the start.",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "end_month": {"type": "integer", "minimum": 1, "maximum": 5},
                    "group_by": {
                        "type": "string",
                        "enum": ["brand", "vehicle_condition", "brand_and_condition", "stage"],
                    },
                },
                "required": ["start_month", "end_month"],
            },
        },
    },
}

PHASE_TOOL_NAMES = {
    "phase_1_portfolio_stability": [
        "get_available_data",
        "get_portfolio_summary",
        "compare_portfolio_totals",
    ],
    "phase_2_segment_screening": [
        "compare_months",
        "get_ecl_movement_by_segment",
    ],
    "phase_3_mechanism_investigation": [
        "get_stage_transition_matrix",
        "get_new_default_analysis",
    ],
    "phase_4_final_synthesis": [],
}

TOOL_CALL_PATTERN = re.compile(
    r"<tool_call\s*>?\s*(\{.*?\})\s*</tool_call\s*>?",
    re.DOTALL,
)


def _extract_balanced_json(text, start_index):
    """Extract one complete JSON object beginning at or after start_index."""
    object_start = text.find("{", start_index)
    if object_start < 0:
        return None

    depth = 0
    in_string = False
    escaped = False

    for index in range(object_start, len(text)):
        character = text[index]

        if in_string:
            if escaped:
                escaped = False
            elif character == "\\":
                escaped = True
            elif character == '"':
                in_string = False
            continue

        if character == '"':
            in_string = True
        elif character == "{":
            depth += 1
        elif character == "}":
            depth -= 1
            if depth == 0:
                return text[object_start:index + 1]

    return None


# -----------------------------------------------------------------------------
# 5. Orchestration helpers
# -----------------------------------------------------------------------------

def _parse_tool_calls(content):
    """Parse complete Qwen tool calls, including calls missing the closing tag."""
    calls = [
        json.loads(match.group(1))
        for match in TOOL_CALL_PATTERN.finditer(content)
    ]
    if calls:
        return calls

    marker_pattern = re.compile(r"<tool_call\s*>?", re.IGNORECASE)
    for marker in marker_pattern.finditer(content):
        payload = _extract_balanced_json(content, marker.end())
        if payload is not None:
            calls.append(json.loads(payload))

    return calls


def _execute_tool(request, allowed_tool_names):
    name = request.get("name", "unknown")
    arguments = request.get("arguments", {})
    if isinstance(arguments, str):
        arguments = json.loads(arguments)

    if name not in allowed_tool_names:
        return name, {
            "error": f"Tool {name!r} is unavailable in this phase.",
            "available_tools": sorted(allowed_tool_names),
        }
    if name not in TOOL_REGISTRY:
        return name, {"error": f"Tool {name!r} is not registered."}

    try:
        result = TOOL_REGISTRY[name](**arguments)
    except Exception as error:
        result = {"error": f"{type(error).__name__}: {error}"}
    return name, result


def _run_tool_phase(
    phase_name,
    phase_prompt,
    allowed_tool_names,
    required_tool_names,
    evidence_log,
    maximum_rounds=6,
):
    """Run one bounded phase and reject completion until its evidence gate is met."""
    phase_tools = [ALL_TOOL_SCHEMAS[name] for name in allowed_tool_names]
    messages = [
        {
            "role": "system",
            "content": (
                "You are a cautious credit-risk analyst. Use only retrieved evidence. "
                "Never invent, rescale or manually calculate a figure. The tools return "
                "both decimal and percentage values where relevant. Prefer the supplied "
                "percentage values. Complete only the current investigation phase."
            ),
        },
        {"role": "user", "content": phase_prompt},
    ]

    used_in_phase = set()
    final_text = ""

    for round_no in range(1, maximum_rounds + 1):
        response = qwen_pipeline(
            messages,
            tools=phase_tools,
            generation_config=AGENT_GENERATION_CONFIG,
        )
        assistant_content = response[0]["generated_text"][-1]["content"] or ""
        messages.append({"role": "assistant", "content": assistant_content})
        tool_calls = _parse_tool_calls(assistant_content)

        if not tool_calls:
            missing = set(required_tool_names) - used_in_phase
            if missing:
                print(
                    f"[{phase_name}, round {round_no}] phase answer rejected; "
                    f"missing required evidence from: {sorted(missing)}"
                )
                messages.append({
                    "role": "user",
                    "content": (
                        "This phase is incomplete. Retrieve the missing evidence using: "
                        f"{sorted(missing)}. Do not provide the phase conclusion yet."
                    ),
                })
                continue

            final_text = assistant_content
            print(f"\n[{phase_name}] PHASE CONCLUSION:\n{final_text}\n")
            return {
                "phase": phase_name,
                "conclusion": final_text,
                "used_tools": sorted(used_in_phase),
                "evidence": deepcopy(evidence_log[phase_name]),
            }

        for request in tool_calls:
            name, result = _execute_tool(request, set(allowed_tool_names))
            if not (isinstance(result, dict) and "error" in result):
                used_in_phase.add(name)
            evidence_item = {
                "tool": name,
                "arguments": request.get("arguments", {}),
                "result": result,
            }
            evidence_log[phase_name].append(evidence_item)
            payload = json.dumps(result)
            print(
                f"[{phase_name}, round {round_no}] {name}"
                f"({json.dumps(request.get('arguments', {}))})\n{payload}\n"
            )
            messages.append({"role": "tool", "name": name, "content": payload})

    raise RuntimeError(
        f"{phase_name} reached its limit of {maximum_rounds} rounds without completion."
    )


def _compact_evidence(evidence_log):
    """Create the final evidence pack without exposing internal shock indicators."""
    return {
        phase: items
        for phase, items in evidence_log.items()
        if phase != "phase_4_final_synthesis"
    }


def _run_final_synthesis(phase_results, evidence_log):
    """Generate an evidence-only final report with no tools available."""
    evidence_pack = _compact_evidence(evidence_log)
    prompt = {
        "task": (
            "Produce the final IFRS 9 portfolio investigation report using only the "
            "verified evidence below. Do not add metrics, abbreviations or explanations "
            "that are absent from the evidence. Do not redo arithmetic."
        ),
        "required_sections": [
            "Overall stability conclusion",
            "Portfolio-level movement",
            "Principal deteriorating segments",
            "Observed deterioration mechanism",
            "Limitations and unverified hypotheses",
        ],
        "phase_conclusions": phase_results,
        "verified_tool_evidence": evidence_pack,
    }

    messages = [
        {
            "role": "system",
            "content": (
                "You are writing an evidence-controlled credit-risk investigation. "
                "Use only the supplied evidence. Copy figures exactly as supplied. "
                "Treat decimal rates and percentage rates as different units. "
                "If evidence does not prove an external cause, say so explicitly."
            ),
        },
        {"role": "user", "content": json.dumps(prompt)},
    ]

    response = qwen_pipeline(
        messages,
        generation_config=AGENT_GENERATION_CONFIG,
    )
    final_answer = response[0]["generated_text"][-1]["content"] or ""
    print("=" * 80)
    print("FINAL FOUR-PHASE INVESTIGATION REPORT")
    print(final_answer)
    return final_answer


# -----------------------------------------------------------------------------
# 6. Four-phase orchestrator
# -----------------------------------------------------------------------------

def run_four_phase_investigation():
    evidence_log = {
        phase: [] for phase in PHASE_TOOL_NAMES
    }
    phase_results = []

    phase_1 = _run_tool_phase(
        phase_name="phase_1_portfolio_stability",
        phase_prompt=(
            "Assess whether the portfolio is stable over the available months. "
            "First confirm the available data. Retrieve all monthly portfolio summaries "
            "and compare the earliest and latest months. Identify which period or periods "
            "warrant deeper investigation. Do not investigate segments yet."
        ),
        allowed_tool_names=PHASE_TOOL_NAMES["phase_1_portfolio_stability"],
        required_tool_names={
            "get_available_data",
            "get_portfolio_summary",
            "compare_portfolio_totals",
        },
        evidence_log=evidence_log,
        maximum_rounds=6,
    )
    phase_results.append(phase_1)

    phase_2 = _run_tool_phase(
        phase_name="phase_2_segment_screening",
        phase_prompt=(
            "The portfolio-level review found instability. Screen the relevant comparison "
            "period for observable segment drivers. Compare brands, vehicle conditions, "
            "brand-condition combinations and stages. Attribute the ECL movement and "
            "identify candidate segments for deeper investigation. Do not claim a root "
            "cause yet. The previous phase conclusion was: "
            + phase_1["conclusion"]
        ),
        allowed_tool_names=PHASE_TOOL_NAMES["phase_2_segment_screening"],
        required_tool_names={"compare_months", "get_ecl_movement_by_segment"},
        evidence_log=evidence_log,
        maximum_rounds=6,
    )
    phase_results.append(phase_2)

    phase_3 = _run_tool_phase(
        phase_name="phase_3_mechanism_investigation",
        phase_prompt=(
            "Investigate the candidate segments identified in the screening phase. Use "
            "stage-transition and new-default evidence to establish the observed mechanism. "
            "Distinguish accounts newly entering default from those already defaulted. "
            "Do not infer an external business cause that is not present in the data. "
            "The screening conclusion was: "
            + phase_2["conclusion"]
        ),
        allowed_tool_names=PHASE_TOOL_NAMES["phase_3_mechanism_investigation"],
        required_tool_names={"get_stage_transition_matrix", "get_new_default_analysis"},
        evidence_log=evidence_log,
        maximum_rounds=6,
    )
    phase_results.append(phase_3)

    final_answer = _run_final_synthesis(phase_results, evidence_log)

    return {
        "phase_results": phase_results,
        "evidence_log": evidence_log,
        "final_answer": final_answer,
    }


# -----------------------------------------------------------------------------
# 7. Execute with a heartbeat
# -----------------------------------------------------------------------------

def _heartbeat(stop_event):
    elapsed = 0
    while not stop_event.wait(5):
        elapsed += 5
        print(
            f"[heartbeat] {elapsed}s elapsed - four-phase investigation is running",
            flush=True,
        )


_stop_event = threading.Event()
_heartbeat_thread = threading.Thread(
    target=_heartbeat,
    args=(_stop_event,),
    daemon=True,
)
_heartbeat_thread.start()

try:
    investigation_output = run_four_phase_investigation()
finally:
    _stop_event.set()
    _heartbeat_thread.join(timeout=1)
